# D42 Pipeline Code

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.09
제작 : 박광석 (모두의연구소)

# 10단계 데이터 전처리 파이프라인 구현 실습

## 실행 환경

**실습 목표**
- `Doc` 데이터 계약과 예외 포착 실행기(`Pipeline.run`), 하드·소프트 검증 게이트를 결합해 교재 4.4절의 10단계 전처리 파이프라인을 구현할 수 있다.
- 청크별 리니지 메타데이터와 문서 단위 라이선스·인코딩 대장을 조인하여 점검하고 0단계 정답 기반 평가 질의 세트(`hit@k`, `hit@1`)로 검색 품질을 검증할 수 있다.
- 스냅샷 비교를 통한 증분 처리·삭제 동기화, 연속 실행 멱등성 검증, 체크포인트 백필·재개(Resume) 및 정기 스케줄 감시를 구현할 수 있다.

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 값 |
|---|---|
| 기준 환경 | 코랩 Pro+. **선택 트랙까지 포함해 전 셀이 GPU 없이 실행됩니다** |
| 필요한 키 | 없음. 외부 API를 호출하지 않습니다 |
| 로드하는 모델 | 기본 0개. 선택 트랙 3과 5에서만 임베딩 모델 1개 (약 0.5GB) |
| 오래 걸리는 셀 | Step 0 설치 2~3분, 선택 트랙 2의 확대 측정 1~2분, 트랙 3의 모델 다운로드 2~3분 |
| 전체 소요 | 4~5시간 (선택 트랙 두 개 포함) |

</div>

오늘 분량은 파일 처리와 정규식과 집계가 대부분이라 GPU를 쓰지 않고 생성 모델도 로드하지 않습니다. **데이터 파이프라인 엔지니어링의 핵심은 모델 호출보다 입출력 계약, 실행 순서, 처리 이력(리니지)의 정밀한 통제에 있습니다.**

## Step 0. 환경 준비

버전을 고정해 설치합니다. 이론 3.2절의 도구들은 배포가 잦고 이론 2.5절의 다섯 번째 항목과 3절 두 번째 Quiz에서 본 대로 **버전이 올라가면 우리 코드가 하나도 안 바뀌어도 결과가 달라집니다.**

메이저 버전에 상한을 두는 방식(`"이름>=x,<y"`)으로 자잘한 고침은 받아들이되 구조가 바뀌는 버전 올림은 막습니다.

In [1]:
import torch
from google.colab import drive
import os

drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aiffel
os.chdir('/content/drive/MyDrive/aiffel')

Mounted at /content/drive


In [2]:
# 문서 처리 쪽
!pip install -q "python-docx>=1.2,<2" "openpyxl>=3.1,<4"
# 정제와 중복 판정 쪽
!pip install -q "chardet>=7,<8" "datasketch>=2,<3" "langdetect>=1.0.9,<2"
# 청킹(Step 9)과 벡터DB(선택 트랙 3, 5) 쪽
!pip install -q "langchain-text-splitters>=1.1,<2" "qdrant-client>=1.19,<2"
!pip install -q "langchain-huggingface>=1.2,<2" "sentence-transformers>=6,<7"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 1.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 4.0 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 3.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 4.2 MB/s eta 0:00:0000:0100:01


설치가 끝나면 코랩이 **런타임을 다시 시작하라**고 알릴 수 있습니다. 알림이 뜨면 다시 시작한 뒤 설치 셀은 건너뛰고 이어서 실행합니다.

In [3]:
# 실제로 설치된 버전을 출력해 둡니다. 결과가 달라졌을 때 가장 먼저 볼 기록입니다
from importlib.metadata import version, PackageNotFoundError

PINNED = ["python-docx", "openpyxl", "chardet", "datasketch", "langdetect",
          "langchain-text-splitters", "qdrant-client"]


def installed_versions(names=PINNED):
    out = {}
    for name in names:
        try:
            out[name] = version(name)
        except PackageNotFoundError:
            out[name] = "(설치되지 않음)"
    return out


VERSIONS = installed_versions()
for k, v in VERSIONS.items():
    print(f"{k:26} {v}")

python-docx                1.2.0
openpyxl                   3.1.5
chardet                    7.7.0
datasketch                 2.0.0
langdetect                 1.0.9
langchain-text-splitters   1.1.3
qdrant-client              1.19.1


`qdrant-client`처럼 **`__version__`을 아예 두지 않는 패키지가 있어서** `__version__` 속성 대신 `importlib.metadata.version()`을 썼습니다.

`VERSIONS` 딕셔너리는 출력하고 버리지 않습니다. 이론 4.2절의 실행 리니지(실행마다 언제 무엇을 어떻게 만들었는지 남기는 기록)대로 Step 9에서 **실행 기록에 함께 저장**합니다.

In [4]:
import os, re, json, hashlib, shutil, time, random
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field, asdict
import numpy as np

random.seed(42)
print("준비 완료")

준비 완료


## Step 1. 사내 문서 저장소 준비

어제와 같은 문서 저장소를 만들고 이미 있으면 그대로 씁니다. 여기서 저장소를 다시 만들므로 어제 노트를 실행하지 않았어도 됩니다.

In [5]:
ROOT = Path("./corp_docs")

if not ROOT.exists():
    print("문서 저장소를 새로 생성합니다")
    for d in ["공용/규정", "공용/매뉴얼", "인사/평가", "영업/계약", "임시"]:
        (ROOT / d).mkdir(parents=True, exist_ok=True)

    docs = {
        "공용/규정/복무규정_2024.md": "# 임직원 복무 규정 (2024년판)\n\n## 제3조 (근무 시간)\n기본 근무 시간은 주 40시간으로 하며, 시업 시각은 오전 9시로 한다.\n\n## 제5조 (재택근무)\n임직원은 주 1회 이내에서 재택근무를 신청할 수 있다.\n\n## 제7조 (연차 휴가의 이월)\n당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없다.\n",
        "공용/규정/복무규정_최종.md": "# 임직원 복무 규정 (2026년판)\n\n## 제3조 (근무 시간)\n기본 근무 시간은 주 40시간으로 하며, 시업 시각은 오전 10시로 한다.\n\n## 제5조 (재택근무)\n임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다.\n\n## 제7조 (연차 휴가의 이월)\n당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 이월할 수 있다.\n",
        "공용/규정/보안지침.md": "# 정보보안 지침\n\n## 1. 사내 데이터의 외부 반출\n고객 개인정보, 미공개 재무 정보, 기술 설계 문서는 정보보호팀장의 사전 승인 없이 외부로 반출할 수 없다. 외부 AI 서비스에 사내 문서를 입력하는 행위도 반출로 간주한다.\n\n## 2. 위반 시 조치\n위반한 임직원은 상벌 규정에 따라 징계할 수 있다.\n",
        "공용/매뉴얼/시스템매뉴얼.md": "# 사내 시스템 사용 매뉴얼\n\n## 1. 계정 발급\n입사 후 IT팀에 계정 발급을 신청한다. 신청은 사내 포털의 IT 요청 메뉴에서 진행한다.\n\n## 2. VPN 접속\n사외에서 사내 시스템에 접속할 때는 VPN을 사용한다.\n\n## 3. 문서 저장 규칙\n업무 문서는 부서별 공유 드라이브에 저장한다.\n",
        "임시/시스템매뉴얼_복사.md": "# 사내 시스템 사용 매뉴얼\n\n## 1. 계정 발급\n입사 후 IT팀에 계정 발급을 신청한다. 신청은 사내 포털의 IT 요청 메뉴에서 진행한다.\n\n## 2. VPN 접속\n사외에서 사내 시스템에 접속할 때는 VPN을 사용한다.\n\n## 3. 문서 저장 규칙\n업무 문서는 부서별 공유 드라이브에 저장한다.\n",
        "인사/평가/인사평가.md": "# 2026년 상반기 인사평가 결과 (대외비)\n\n1. 김서준 - 주민등록번호 880315-1234567, 연락처 010-2345-6789, 평가 A\n2. 이지우 - 주민등록번호 920708-2345678, 연락처 010-3456-7890, 평가 B\n",
        "영업/계약/계약현황.md": "# 협력사 계약 현황\n\n1. 한빛물산 - 계약금액 48,000,000원, 납기 2026년 11월 30일, 담당 김서준 010-2345-6789\n2. 대성테크 - 계약금액 32,000,000원, 납기 2026년 12월 15일, 담당 최유나\n",
        "임시/신규문서.md": "",
        "임시/템플릿.md": "# 제목\n\n작성자 :\n작성일 :\n부서 :\n\n내용을 입력하세요.\n",
    }
    for p, t in docs.items():
        (ROOT / p).write_text(t, encoding="utf-8")

    from docx import Document as Docx
    d = Docx()
    d.add_heading("출장 여비 규정", 0)
    d.add_paragraph("국내 출장 시 일비 3만원, 숙박비 실비를 지급한다.")
    d.add_paragraph("해외 출장 시 지역별 기준에 따라 차등 지급한다.")
    d.save(ROOT / "공용/규정/출장여비규정.docx")

    (ROOT / "임시/깨진파일.txt").write_bytes("사내 규정 안내 문서입니다".encode("euc-kr"))

print(f"문서 {sum(1 for _ in ROOT.rglob('*') if _.is_file())}건")

문서 19건


## Step 2. 파이프라인 실행기 및 문서 계약(`Doc`) 정의

이론 1.5절에서 정의한 다섯 가지 핵심 개념(멱등성, 재개, 리니지, 증분 처리, 검증 게이트)을 실제 파이썬 코드 구조로 구현하며 전체 단계 구성은 교재 4.4절의 전처리 10단계(데이터 엔지니어링 노트 1.4절)를 따릅니다.

<div style="font-size: 18px; line-height: 1.6;">

| 이론 1.5절 용어 | 이 노트에서 만드는 것 | 실습 위치 |
|---|---|---|
| 멱등성 | 내용에서 계산한 결정적 식별자 | Step 3, Step 9, Step 12 |
| 재개 | 중간 산출물 저장과 `start_from` | Step 2, Step 13, Step 13-1 |
| 리니지 | `Doc.history`와 청크에 기록하는 여덟 항목 | Step 2, Step 9 |
| 증분 처리 | 스냅샷 대조와 조정 | Step 10, Step 11 |
| 검증 게이트 | 하드 게이트와 소프트 게이트 | Step 7, Step 8 |

</div>

다섯 가지 장치를 올리기 전에 먼저 표의 바탕이 되는 **단계 간 계약(`Doc` 데이터 클래스)**부터 정의합니다(이론 1.5절). 모든 전처리 단계 함수가 `list[Doc]`을 입력받아 `list[Doc]`을 반환하도록 입출력 계약을 통일하고 각 문서 객체 내부의 `history` 리스트에 단계별 처리 이력을 누적하면, 실행 순서 제어와 건수 계측, 체크포인트 저장, 데이터 리니지 기록이 한꺼번에 해결됩니다.

In [6]:
@dataclass
class Doc:
    """파이프라인을 흐르는 단위. 리니지를 스스로 들고 다닙니다"""
    doc_id: str
    path: str
    ext: str = ""
    size: int = 0
    mtime: float = 0.0
    collected_at: float = 0.0        # 우리가 이 문서를 발견한 시각 (이론 4.3절)
    dept: str = ""
    access_level: str = ""
    file_hash: str = ""

    text: str = ""
    lang: str = ""
    policy: str = ""

    status: str = "pending"          # pending, ok, excluded, failed
    reject_reason: str = ""
    retries: int = 0
    history: list = field(default_factory=list)   # 리니지

    def log(self, stage, note=""):
        self.history.append({"stage": stage, "note": note, "at": time.time()})

    def exclude(self, stage, reason):
        self.status = "excluded"
        self.reject_reason = reason
        self.log(stage, f"제외: {reason}")

    def fail(self, stage, reason):
        self.status = "failed"
        self.reject_reason = reason
        self.log(stage, f"실패: {reason}")

`collected_at`은 **파이프라인이 문서를 수집한 시각**이고 `mtime`은 **원본 파일이 마지막으로 수정된 시각**이므로 두 필드는 서로 다른 질문에 답합니다. 이론 5.5절에서 지연 도착 문서를 놓치지 않기 위해 문서 내부 날짜 대신 수집 시각을 기록하라고 한 필드가 `collected_at`이고 이론 4.3절에서 "검색된 청크가 최신 버전인가"를 판별하는 핵심 단서로 꼽은 필드가 `mtime`입니다.

In [7]:
@dataclass
class StageResult:
    name: str
    n_in: int
    n_out: int
    n_excluded: int
    n_failed: int
    seconds: float
    reasons: dict = field(default_factory=dict)
    warnings: list = field(default_factory=list)


class Pipeline:
    """단계를 순서대로 실행하고 계측과 게이트를 관리합니다"""

    def __init__(self, stage_dir="./stage"):
        self.stages = []
        self.results = []
        self.stopped_at = ""          # 멈춘 단계 이름. 재개할 때 여기부터 시작합니다
        self.stage_dir = Path(stage_dir)
        self.stage_dir.mkdir(exist_ok=True)

    def add(self, name, fn, gate=None, soft=None, save=False):
        """gate: 하드 게이트. soft: 소프트 게이트. 둘 다 (docs) -> (ok, message)"""
        self.stages.append({"name": name, "fn": fn, "gate": gate,
                            "soft": soft, "save": save})
        return self

    def run(self, docs, start_from=None, verbose=True):
        self.stopped_at = ""
        started = start_from is None
        for st in self.stages:
            if not started:
                if st["name"] == start_from:
                    started = True
                else:
                    if verbose:
                        print(f"[건너뜀] {st['name']}")
                    continue

            alive_in = [d for d in docs if d.status in ("pending", "ok")]
            t0 = time.time()
            # 단계가 통째로 실패해도 예외를 위로 던지지 않습니다.
            # 여기까지의 산출물을 보존하고 어느 단계에서 중단되었는지 반환합니다
            try:
                docs = st["fn"](docs)
            except Exception as e:
                self.stopped_at = st["name"]
                print(f"[{st['name']}] 단계가 중단되었습니다 : {type(e).__name__} {e}")
                return docs, False
            elapsed = time.time() - t0

            alive_out = [d for d in docs if d.status in ("pending", "ok")]
            excluded = [d for d in docs if d.status == "excluded"]
            failed = [d for d in docs if d.status == "failed"]

            res = StageResult(
                name=st["name"], n_in=len(alive_in), n_out=len(alive_out),
                n_excluded=len(excluded), n_failed=len(failed), seconds=elapsed,
                reasons=dict(Counter(d.reject_reason.split(":")[0]
                                     for d in excluded + failed if d.reject_reason)),
            )
            self.results.append(res)

            if verbose:
                print(f"[{st['name']:<16}] 입력 {res.n_in:>3} -> 출력 {res.n_out:>3} "
                      f"(제외 {res.n_excluded}, 실패 {res.n_failed}) {res.seconds:.2f}초")

            # 소프트 게이트 : 경고만 남기고 계속 갑니다
            if st["soft"]:
                _, warn = st["soft"](docs)
                if warn:
                    res.warnings.append(warn)
                    print(f"  [경고] {warn}")

            # 하드 게이트 : 걸리면 멈춥니다
            if st["gate"]:
                ok, msg = st["gate"](docs)
                if not ok:
                    self.stopped_at = st["name"]
                    print(f"\n  게이트 실패 : {msg}")
                    print("  파이프라인을 중단합니다")
                    return docs, False

            # 중간 산출물 저장
            if st["save"]:
                self.save_stage(st["name"], docs)

        return docs, True

    def save_stage(self, name, docs):
        out = self.stage_dir / f"{name}.json"
        with open(out, "w", encoding="utf-8") as f:
            json.dump([asdict(d) for d in docs], f, ensure_ascii=False)

    def load_stage(self, name):
        p = self.stage_dir / f"{name}.json"
        rows = json.load(open(p, encoding="utf-8"))
        return [Doc(**r) for r in rows]

    def report(self):
        print(f"{'단계':<18} {'입력':>6} {'출력':>6} {'제외':>6} {'실패':>6} {'초':>7}")
        print("-" * 56)
        for r in self.results:
            print(f"{r.name:<18} {r.n_in:>6} {r.n_out:>6} {r.n_excluded:>6} "
                  f"{r.n_failed:>6} {r.seconds:>7.2f}")

`Doc` 클래스는 내부에 `history` 리스트를 보유하여 파이프라인의 각 단계를 통과할 때마다 **어느 단계에서 어떤 처리를 받았는지 스스로 이력을 기록**합니다(이론 4.2절의 3계층 중 데이터 리니지).

또한 `Pipeline.run` 메서드는 단계 실행 중 발생한 예외를 잡아 `(docs, False)` 형태로 반환합니다. 예외를 상위 호출자로 그대로 발생시키면(raise) 노트북 셀 실행이 에러와 함께 중단되면서 **이전 단계까지 처리한 중간 산출물 변수도 함께 유실되어** 이론 1.2절에서 살펴본 전체 재실행 문제가 발생하기 때문입니다.

`start_from` 인자는 지정한 단계 이전의 단계들을 건너뛰고 저장된 중간 산출물부터 이어서 실행하는 장치로 이론 2.4절의 **재개(Resume)**와 **백필(Backfill)**을 담당합니다(Step 13과 Step 13-1에서 직접 검증합니다).

## Step 3. 1단계 데이터 목록화

교재 : 4.4.1 데이터 목록화하기

내용을 읽지 않고 메타데이터만 수집합니다. **doc_id를 결정적으로 만드는 것**이 멱등성의 출발점입니다.

교재 4.4.1절에서는 프로젝트 목표와 용도, 데이터 형식·언어·도메인·품질 기준을 먼저 정의하고 데이터 출처를 카탈로그 DB로 정리해 태그를 부여하도록 안내합니다. 아래 문서 목록과 `DEPT_MAP`(폴더 → 부서·등급 태그)이 그 출처 데이터베이스의 가장 단순한 형태이고 품질과 언어 기준은 Step 4와 Step 5에 있습니다.

**[현업 환경 적용]** : 여기서는 `./corp_docs`라는 로컬 폴더를 훑지만 사내에서는 마운트된 공유 드라이브나 문서 관리 시스템의 목록 기능을 씁니다. 그러면 **드라이브가 연결되지 않았을 때 목록이 0건이 되고 오류는 나지 않습니다.** 이론 6.3절에서 0건은 기본적으로 실패로 다루라고 한 상황이 여기서 생기므로 Step 8에서 이 단계에 0건 게이트를 붙입니다.

In [9]:
import hashlib
import time
import unicodedata  # 💡 1. unicodedata 모듈 임포트

DEPT_MAP = {
    "공용": ("ALL", "public"),
    "인사": ("HR", "confidential"),
    "영업": ("SALES", "internal"),
    "임시": ("ALL", "draft"),
}


def stage_catalog(_docs=None):
    now = time.time()
    docs = []
    # sorted 로 훑습니다. 순회 순서가 결과에 영향을 주면 멱등성이 깨집니다 (이론 2.5절)
    for p in sorted(ROOT.rglob("*")):
        if not p.is_file():
            continue

        # 💡 2. [수정] 상대 경로를 가져올 때 바로 NFC(완성형)로 정규화합니다.
        # .as_posix()를 붙이면 윈도우(\)와 리눅스(/) 구분자도 '/'로 통일되어 split("/")이 안전해집니다.
        rel = unicodedata.normalize("NFC", p.relative_to(ROOT).as_posix())

        raw = p.read_bytes()
        fh = hashlib.sha256(raw).hexdigest()

        # 멱등성 : 경로와 내용으로 ID를 만듭니다. 같은 문서는 항상 같은 ID
        doc_id = hashlib.sha256(f"{rel}|{fh}".encode()).hexdigest()[:16]

        # rel이 이미 NFC이므로 rel.split("/")[0]도 정상적인 완성형("공용")이 됩니다!
        top_folder = rel.split("/")[0]
        dept, level = DEPT_MAP.get(top_folder, ("UNKNOWN", "unknown"))

        d = Doc(
            doc_id=doc_id,
            path=rel,
            ext=p.suffix.lower(),
            size=len(raw),
            mtime=p.stat().st_mtime,
            collected_at=now,
            dept=dept,
            access_level=level,
            file_hash=fh[:16],
            status="pending",
        )
        d.log("catalog", f"수집 대상 등록 ({dept}/{level})")
        docs.append(d)
    return docs


docs = stage_catalog()
print(f"{len(docs)}건 목록화")
for d in docs[:5]:
    print(f"  {d.doc_id}  {d.path:<34} {d.dept:>6}/{d.access_level}")

19건 목록화
  be7be43ebd4ac223  공용/규정/공문처리지침.hwpx                     ALL/public
  38c13db0123a599e  공용/규정/교육비지원규정.pdf                     ALL/public
  d93e15d0576c923c  공용/규정/보안지침_1월.md                      ALL/public
  c0601ffe080cdf07  공용/규정/보안지침_7월.md                      ALL/public
  d6444204d636dfb1  공용/규정/복무규정_2024.md                    ALL/public


`doc_id`는 경로와 내용 해시로 만들어져 같은 문서를 다시 처리하면 같은 ID가 나옵니다. 벡터DB에 넣을 때 이 ID를 쓰면 중복 삽입 대신 덮어쓰기가 됩니다. 이론 2.3절의 첫 번째 장치입니다.

**`collected_at`은 ID 계산에 넣지 않았습니다.** 이론 2.5절에서 멱등성이 조용히 깨지는 경우 첫째로 꼽은 것이 식별자에 현재 시각을 섞는 일입니다. 시각은 리니지에는 필요하지만 식별자에 넣으면 멱등성이 깨지므로 두 용도를 분명히 나눕니다.

**내용이 바뀌면 ID도 바뀌는 것**은 의도된 동작입니다. 이전 버전의 청크를 지우고 새로 넣어야 하고 그 처리는 Step 10에서 다룹니다.

## Step 4. 2단계 규정 준수 확인과 3단계 필터링

교재 : 4.4.2 프라이버시 및 법적 규정 준수 확인하기, 4.4.3 데이터 필터링하기

데이터 엔지니어링 노트에서 만든 로직을 파이프라인 단계 함수로 옮깁니다. 교재 4.4.2절에서는 개인정보 및 법적 규정 준수 점검과 함께 문서별 라이선스를 확인해 기록하도록 안내합니다. 라이선스 기록은 Step 9에서 청크 점검과 함께 다룹니다.

In [10]:
import chardet

PII_PATTERNS = {
    "주민등록번호": r"\b\d{6}\s?-\s?[1-4]\d{6}\b",
    "휴대전화": r"\b01[016-9]\s?-?\s?\d{3,4}\s?-?\s?\d{4}\b",
    "계좌번호": r"\b(?!01[016-9]-)\d{2,3}-\d{2,6}-\d{4,8}\b",
    "이메일": r"\b[\w.+-]+@[\w-]+\.[\w.]+\b",
}
HIGH_RISK = {"주민등록번호", "계좌번호"}
# 이론 6.2절 Fail-Closed 원칙: 사전에 정의된 보안 등급만 허용하고 미정의 등급(unknown, secret 등)은 즉시 격리합니다
KNOWN_ACCESS_LEVELS = {"public", "draft", "internal", "confidential"}


def read_bytes_text(p: Path):
    raw = p.read_bytes()
    if not raw:
        return "", "empty"
    enc = chardet.detect(raw)["encoding"] or "utf-8"
    try:
        return raw.decode(enc, errors="replace"), enc
    except (LookupError, UnicodeDecodeError):
        return raw.decode("utf-8", errors="replace"), "utf-8-fallback"


def detect_pii(text):
    return {k for k, v in PII_PATTERNS.items() if re.search(v, text)}


def stage_compliance(docs):
    for d in docs:
        if d.status not in ("pending", "ok"):
            continue
        # 1) 스키마 드리프트 방어 (Fail-Closed): 정의되지 않은 보안 등급이나 폴더 구조 변경 시 즉시 제외
        if d.access_level not in KNOWN_ACCESS_LEVELS:
            d.exclude("compliance", f"스키마 드리프트(미정의 보안 등급 Fail-Closed 격리): {d.access_level}")
            continue
        if d.access_level == "confidential":
            d.exclude("compliance", "권한: 기밀 등급 문서")
            continue
        if d.ext not in (".md", ".txt"):
            d.policy = ""                 # 아직 정하지 않음. 파싱 뒤에 정합니다
            d.log("compliance", "형식상 사전 검사 보류 (파싱 후 재검사)")
            continue

        text, _ = read_bytes_text(ROOT / d.path)
        pii = detect_pii(text)

        high = HIGH_RISK & pii
        if high:
            d.exclude("compliance", f"고위험PII: {', '.join(sorted(high))}")
            continue
        d.policy = "mask" if pii else "include"
        d.log("compliance", f"방침 {d.policy} (탐지 {sorted(pii)})")
    return docs


# 스키마 드리프트(신규 미정의 등급 'secret_tier' 유입) Fail-Closed 동작 검증
drift_doc = Doc(doc_id="drift001", path="신규부서/극비프로젝트.md", ext=".md", size=120,
                mtime=time.time(), collected_at=time.time(), dept="NEW", access_level="secret_tier")
stage_compliance([drift_doc])
print(f"[Fail-Closed 검증] 미정의 등급 문서 상태: {drift_doc.status} / 사유: {drift_doc.reject_reason}")

[Fail-Closed 검증] 미정의 등급 문서 상태: excluded / 사유: 스키마 드리프트(미정의 보안 등급 Fail-Closed 격리): secret_tier


`.md`와 `.txt`가 아닌 형식은 **본문을 보려면 먼저 파싱해야 하므로** 여기서 방침을 정하지 못합니다. 그래서 방침을 빈 문자열로 두고 파싱 뒤에 다시 검사합니다.

이 재검사 단계를 빠뜨리면 주민등록번호가 든 `.docx` 하나가 아무 검사도 받지 않고 인덱스까지 갑니다. **검사를 미루는 것은 괜찮고 미룬 것을 잊는 것이 사고입니다.**

In [11]:
def stage_compliance_post(docs):
    """파싱해야 본문이 보이는 형식을 뒤늦게 다시 검사합니다"""
    for d in docs:
        if d.status not in ("pending", "ok") or d.policy:
            continue                      # 이미 방침이 정해진 것은 건드리지 않습니다
        pii = detect_pii(d.text)
        high = HIGH_RISK & pii
        if high:
            d.exclude("compliance_post", f"고위험PII: {', '.join(sorted(high))}")
            continue
        d.policy = "mask" if pii else "include"
        d.log("compliance_post", f"파싱 후 재검사, 방침 {d.policy}")
    return docs

In [12]:
MIN_CHARS = 50


def stage_filter(docs):
    for d in docs:
        if d.status not in ("pending", "ok"):
            continue
        text = d.text or ""

        # 빈 본문도 통과시키면 안 됩니다. 건너뛰면 status 가 pending 인 채로
        # 살아남아 뒤에서 조용히 ok 가 됩니다
        if len(text.strip()) < MIN_CHARS:
            d.exclude("filter", f"본문부족: {len(text.strip())}자")
            continue

        lines = [l.strip() for l in text.split("\n") if l.strip()]
        boiler = sum(1 for l in lines if l.endswith(":") or l.endswith("："))
        if lines and boiler / len(lines) > 0.3:
            d.exclude("filter", f"상투적문구(보일러플레이트): {boiler}/{len(lines)}")
            continue

        bad = text.count("\ufffd")
        if bad / max(len(text), 1) > 0.05:
            d.exclude("filter", f"인코딩깨짐: {bad/len(text):.0%}")
            continue

        d.log("filter", "통과")
    return docs

`stage_filter` 함수에서는 본문이 비어 있는 문서를 단순 건너뛰기(`continue`)하지 않고 **명시적으로 제외(`exclude`) 처리**합니다. 만약 빈 문서를 건너뛰어 버리면 해당 문서는 초기 `pending` 상태 그대로 뒷단 단계를 통과해 마지막에 아무런 오류나 경고 없이 정상(`ok`) 문서로 둔갑하므로(이론 6절 무음 장애), 모든 판정 함수의 기본값은 통과가 아닌 불합격이어야 합니다.

또한 교재 4.4.3절에서 안내하는 것처럼 걸러낸 데이터를 물리적으로 삭제하는 대신 `'제외됨(excluded)'` 상태와 사유를 남겨 두면 약간의 저장 공간만 쓰고도 추후 필터 기준을 완화하거나 감사 추적을 수행하기가 훨씬 수월해집니다.

## Step 5. 5단계 텍스트 수집·6단계 인코딩·7단계 언어 감지

교재 : 4.4.5 데이터 수집하기, 4.4.6 인코딩 감지하기, 4.4.7 언어 감지하기

필터링이 본문 길이를 보므로 파싱이 먼저 필요합니다. **그래서 실제 순서는 목록화 → 준수 확인 → 수집 → 필터링입니다.** 데이터 엔지니어링 노트 1.4절에 적힌 대로 **번호는 개념의 순서이지 코드의 실행 순서가 아닙니다.**

교재 4.4.5절에서는 웹 크롤러·API 수집 시 서비스 약관과 저작권 확인을 강조하고 4.4.6~4.4.7절에서는 감지된 인코딩과 언어를 메타데이터에 추가하도록 안내합니다. 이 노트에서 언어는 `d.lang` 필드에 저장되지만 인코딩은 처리 이력(`history`)의 로그 문자열 안에만 남습니다. 이 차이는 Step 9에서 확인합니다.

In [13]:
def parse_md_txt(p):
    text, enc = read_bytes_text(p)
    return text, enc


def parse_docx(p):
    from docx import Document
    d = Document(p)
    parts = [x.text for x in d.paragraphs if x.text.strip()]
    for t in d.tables:
        for row in t.rows:
            parts.append(" | ".join(c.text for c in row.cells))
    return "\n".join(parts), "docx"


def parse_xlsx(p):
    import openpyxl
    wb = openpyxl.load_workbook(p, data_only=True)
    parts = []
    for ws in wb.worksheets:
        parts.append(f"[시트: {ws.title}]")
        for row in ws.iter_rows(values_only=True):
            cells = [str(c) for c in row if c is not None]
            if cells:
                parts.append(" | ".join(cells))
    return "\n".join(parts), "xlsx"


PARSERS = {".md": parse_md_txt, ".txt": parse_md_txt,
           ".docx": parse_docx, ".xlsx": parse_xlsx}

# 다시 해볼 만한 실패 (이론 2.4절). 파일이 잠겼거나 잠깐 못 읽은 경우입니다
RETRYABLE = (OSError, TimeoutError)


def stage_ingest(docs, retries=2, wait=0.05):
    for d in docs:
        if d.status not in ("pending", "ok"):
            continue
        fn = PARSERS.get(d.ext)
        if fn is None:
            d.exclude("ingest", f"미지원형식: {d.ext}")
            continue

        for attempt in range(retries + 1):
            try:
                text, enc = fn(ROOT / d.path)
                d.text = text
                d.log("ingest", f"파싱 완료 ({enc}, {len(text)}자)")
                break
            except RETRYABLE as e:
                d.retries += 1
                if attempt < retries:
                    d.log("ingest", f"재시도 {d.retries}회 ({type(e).__name__})")
                    time.sleep(wait)
                    continue
                d.fail("ingest", f"읽기오류: {type(e).__name__} (재시도 {d.retries}회 실패)")
                break
            except Exception as e:
                # 형식이 안 맞아 실패하는 것은 백 번을 다시 해도 똑같이 실패합니다
                d.fail("ingest", f"형식오류: {type(e).__name__}")
                break
    return docs

이론 2.4절과 2절 두 번째 Quiz에서 다룬 것처럼 **일시적 장애로 재시도할 실패(`IOError`, 파일 잠금 등)와 재시도해도 결과가 같은 영구적 실패(지원하지 않는 형식, 파일 손상)**를 코드에서 분리했습니다. 영구적 실패까지 무조건 3회 재시도하면 파이프라인 실행 시간만 배수로 늘어나며 재시도가 발생한 횟수(`d.retries`)도 반드시 함께 기록해야 일시적 장애가 누적되는 징후를 포착할 수 있습니다.

In [14]:
def stage_language(docs):
    from langdetect import detect, LangDetectException
    for d in docs:
        if d.status not in ("pending", "ok") or not d.text.strip():
            continue
        try:
            d.lang = detect(d.text[:1000])
        except LangDetectException:
            d.lang = "unknown"
        d.log("language", f"언어 {d.lang}")
        if d.lang not in ("ko", "en", "unknown"):
            d.exclude("language", f"대상외언어: {d.lang}")
    return docs

## Step 6. 4단계 완전·근사 중복 제거

교재 : 4.4.4 데이터 중복 제거하기

데이터 엔지니어링 노트 1.4절의 **'연산 비용이 큰 작업 앞에 비용이 낮은 판단을 배치한다'**는 원칙에 따라 문서 단위 중복 제거를 청킹 및 임베딩 단계 앞에 배치했습니다. 중복 문서를 그대로 둔 채 청킹하고 임베딩까지 계산한 뒤에 걸러내면 그만큼의 연산 시간과 비용이 낭비되기 때문입니다.

교재 4.4.4절의 중복 제거 기법 비교와 임계값 설정(통상 `0.8` 출발)은 어제 데이터 엔지니어링 노트 4절과 Step 7-1에서 다뤘습니다. 본 파이프라인에서는 정규화 본문 해시(`SHA-256`)로 완전 중복을, `MinHashLSH`(임계값 `0.85`)로 근사 중복을 검출하되 물리적으로 삭제하지 않고 제외 상태(`excluded`)와 상세 사유를 기록합니다.

In [15]:
from datasketch import MinHash, MinHashLSH


def shingles(text, k=3):
    t = re.sub(r"\s+", " ", text)
    return {t[i:i+k] for i in range(len(t) - k + 1)}


def make_minhash(text, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for s in sorted(shingles(text)):
        m.update(s.encode())
    return m


def pick_representative(group):
    def score(d):
        s = 10 if not d.path.startswith("임시") else 0
        if re.search(r"(final|최종|승인)", d.path, re.I):
            s += 5
        # 마지막 기준으로 경로를 넣어 동점을 없앱니다 (이론 2.5절 셋째 항목)
        return (s, d.mtime, len(d.text), d.path)
    return max(group, key=score)


def stage_dedup(docs, near_threshold=0.85):
    alive = [d for d in docs if d.status in ("pending", "ok") and d.text]

    # 완전 중복
    by_hash = {}
    for d in alive:
        h = hashlib.sha256(d.text.encode()).hexdigest()
        by_hash.setdefault(h, []).append(d)

    for h, group in by_hash.items():
        if len(group) < 2:
            continue
        rep = pick_representative(group)
        for d in group:
            if d is not rep:
                d.exclude("dedup", f"완전중복: 대표 {rep.path}")

    # 근사 중복
    alive = [d for d in docs if d.status in ("pending", "ok") and d.text]
    lsh = MinHashLSH(threshold=near_threshold, num_perm=128)
    sigs = {}
    for d in alive:
        m = make_minhash(d.text)
        sigs[d.doc_id] = m
        lsh.insert(d.doc_id, m)

    handled = set()
    for d in alive:
        if d.doc_id in handled:
            continue
        cands = [c for c in lsh.query(sigs[d.doc_id]) if c != d.doc_id]
        if not cands:
            continue
        group = [d] + [x for x in alive if x.doc_id in cands]
        rep = pick_representative(group)
        for x in group:
            handled.add(x.doc_id)
            if x is not rep:
                sim = sigs[x.doc_id].jaccard(sigs[rep.doc_id])
                x.exclude("dedup", f"근사중복: 대표 {rep.path} (유사도 {sim:.2f})")

    for d in docs:
        if d.status == "pending":
            d.status = "ok"
            d.log("dedup", "고유 문서")
    return docs

`pick_representative` 함수에서 우선순위 점수 튜플의 마지막 원소로 파일 경로(`d.path`)를 포함시켜 이론 2.5절의 세 번째 비결정성 요인인 **동점(Tie-breaking) 처리 미지정 문제**를 차단했습니다. 파일 경로는 저장소 안에서 항상 유일하므로 점수 동점이 발생하지 않아 실행할 때마다 동일한 대표 문서가 선택됩니다.

또한 `make_minhash` 함수에서도 글자 조각 집합을 `sorted()`로 정렬해 순회합니다. 파이썬 `set`은 실행마다 순회 순서가 달라질 수 있으므로 **집합을 순회할 때는 항상 정렬을 고정하는 습관**을 코드에 유지합니다.

## Step 7. 하드·소프트 검증 게이트 구현

이론 6.4절의 두 종류인 걸리면 멈추는 **하드 게이트**와 경고만 남기고 지나가는 **소프트 게이트**를 다 만듭니다.

소프트 게이트가 비교할 기준선으로 이론 4.2절의 **실행 리니지**, 곧 실행할 때마다 남기는 실행 기록이 필요합니다. 파일 하나면 됩니다.

In [16]:
RUN_DIR = Path("./runs")
RUN_DIR.mkdir(exist_ok=True)


def save_run_record(pipe, docs, chunks=None, config=None):
    """이 실행이 무엇을 어떻게 만들었는지 하나의 JSON 레코드로 묶어 저장합니다 (이론 4.2절)"""
    seq = len(list(RUN_DIR.glob("run_*.json")))
    out = RUN_DIR / f"run_{seq:04d}_{time.strftime('%Y%m%d_%H%M%S')}.json"
    rec = {
        "finished_at": time.time(),
        "stopped_at": pipe.stopped_at,
        "stages": {r.name: {"n_in": r.n_in, "n_out": r.n_out,
                            "n_excluded": r.n_excluded, "n_failed": r.n_failed,
                            "seconds": round(r.seconds, 3),
                            "reasons": r.reasons, "warnings": r.warnings}
                   for r in pipe.results},
        "n_docs": len(docs),
        "n_ok": sum(1 for d in docs if d.status == "ok"),
        "n_chunks": len(chunks) if chunks is not None else None,
        "config": config or {},
        "versions": VERSIONS,          # 쓴 도구 버전 번호
    }
    with open(out, "w", encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False, indent=2)
    return out


def load_last_run():
    files = sorted(RUN_DIR.glob("run_*.json"))
    return json.load(open(files[-1], encoding="utf-8")) if files else None

실행 기록에 `versions`를 함께 넣었습니다. 이론 3절 두 번째 Quiz는 파싱 도구의 새 버전이 자동으로 설치되면서 결과가 달라졌는데 아무도 몰랐던 상황이었습니다. 조치의 첫째가 버전 고정, 둘째가 **실행할 때 실제로 설치된 버전 번호를 기록에 남기는 것**이었습니다.

In [17]:
# 실습 : 단계 사이를 지키는 하드/소프트 게이트 함수들을 완성하세요

def gate_not_empty(docs):
    """살아 있는 문서가 0건이면 실패 (하드)"""
    alive = [d for d in docs if d.status in ("pending", "ok")]
    if not alive:
        return False, "통과(생존) 문서가 0건입니다"
    return True, ""


def gate_pass_rate(min_rate=0.3):
    """통과율이 min_rate 미만이면 실패하는 게이트를 반환하세요 (하드)"""
    def g(docs):
        alive = sum(1 for d in docs if d.status in ("pending", "ok"))
        ### START CODE HERE ### (생존 문서 비율 계산 1줄)
        rate = alive / max(len(docs), 1)
        ### END CODE HERE ###
        if rate is None or rate < min_rate:
            return False, f"통과율 {(rate or 0):.0%} 가 기준 {min_rate:.0%} 미만입니다"
        return True, ""
    return g


def gate_unique_ids(docs):
    """같은 doc_id 가 두 번 이상 나오면 실패 (하드) — 멱등성 위반 신호"""
    ### START CODE HERE ### (2번 이상 등장한 doc_id 리스트 추출 1줄)
    dup = [k for k, v in Counter(d.doc_id for d in docs).items() if v > 1]
    ### END CODE HERE ###
    if dup:
        return False, f"doc_id 중복 {len(dup)}건. 멱등성 위반 가능성"
    return True, ""


def soft_gate_drift(stage_name, prev_run, tolerance=0.3):
    """직전 실행의 같은 단계와 출력 건수를 비교해 경고만 남기는 게이트 (소프트)"""
    def g(docs):
        if not prev_run or stage_name not in prev_run.get("stages", {}):
            return True, ""                       # 기준선이 없으면 비교하지 않습니다
        before = prev_run["stages"][stage_name]["n_out"]
        now = sum(1 for d in docs if d.status in ("pending", "ok"))
        if before == 0:
            return True, ""
        delta = (now - before) / before
        if abs(delta) > tolerance:
            return True, (f"{stage_name} 출력 {before} -> {now} "
                          f"({delta:+.0%}). 제외 사유별 건수를 먼저 확인하세요")
        return True, ""
    return g

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) gate_pass_rate 내부 통과율 계산
rate = alive / max(len(docs), 1)

# 2) gate_unique_ids 내부 중복 doc_id 검출
dup = [k for k, v in Counter(d.doc_id for d in docs).items() if v > 1]
```

`gate_pass_rate`와 `soft_gate_drift`는 임계값이나 직전 실행 기준선을 인자로 받아 검증 함수를 반환하는 팩토리(Factory) 구조여서 단계마다 서로 다른 기준을 깔끔하게 적용할 수 있습니다.

`soft_gate_drift`는 파이프라인을 중단시키지 않고 경고만 남기는 소프트 게이트이므로 **항상 `ok=True`를 반환**합니다. 대신 이론 6.5절의 원칙에 따라 경고 메시지 끝에 **`"제외 사유별 건수를 먼저 확인하세요"`라는 1차 조치 안내**를 함께 포함시켰습니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 8. 전처리 파이프라인 조립 및 실행

단계들을 순서대로 붙이고 중간 산출물 저장 지점을 지정합니다.

이 셀에서 가장 중요한 판단은 **어느 게이트를 하드로, 어느 게이트를 소프트로 둘지**입니다. 이론 6.4절의 기준은 틀린 데이터가 인덱스에 들어가는 것과 인덱스가 갱신되지 않는 것 중 무엇이 더 나쁜가 하나였습니다. 사내 규정 챗봇은 틀린 데이터가 들어가는 쪽이 더 나쁘므로 하드를 많이 둡니다.

In [18]:
PREV = load_last_run()          # 첫 실행이면 None 입니다

pipe = Pipeline(stage_dir="./stage")

(pipe
 .add("catalog", lambda _: stage_catalog(),
      gate=gate_not_empty,                              # 0건은 실패 (이론 6.3절)
      soft=soft_gate_drift("catalog", PREV))            # 건수 급변은 경고
 .add("compliance", stage_compliance, gate=gate_pass_rate(0.3))
 .add("ingest", stage_ingest, gate=gate_not_empty)
 .add("compliance_post", stage_compliance_post)
 .add("filter", stage_filter,
      gate=gate_pass_rate(0.3),
      soft=soft_gate_drift("filter", PREV))
 .add("language", stage_language)
 .add("dedup", stage_dedup, gate=gate_unique_ids, save=True))

docs, ok = pipe.run([])
print(f"\n파이프라인 완료 : {ok}" + (f" (중단 단계 {pipe.stopped_at})" if not ok else ""))

[catalog         ] 입력   0 -> 출력  19 (제외 0, 실패 0) 0.04초
[compliance      ] 입력  19 -> 출력  18 (제외 1, 실패 0) 0.15초
[ingest          ] 입력  18 -> 출력  15 (제외 4, 실패 0) 0.57초
[compliance_post ] 입력  15 -> 출력  15 (제외 4, 실패 0) 0.00초
[filter          ] 입력  15 -> 출력  13 (제외 6, 실패 0) 0.00초
[language        ] 입력  13 -> 출력  12 (제외 7, 실패 0) 0.27초
[dedup           ] 입력  12 -> 출력   9 (제외 10, 실패 0) 0.04초

파이프라인 완료 : True


In [19]:
pipe.report()

단계                     입력     출력     제외     실패       초
--------------------------------------------------------
catalog                 0     19      0      0    0.04
compliance             19     18      1      0    0.15
ingest                 18     15      4      0    0.57
compliance_post        15     15      4      0    0.00
filter                 15     13      6      0    0.00
language               13     12      7      0    0.27
dedup                  12      9     10      0    0.04


In [21]:
# 제외 사유 상세 및 데드레터 큐(quarantine/) 격리·웹훅 알림 페이로드 생성 (이론 4.1·6.4·6.5절)
print(f"{'경로':<36} {'상태':<10} 사유")
print("-" * 84)
for d in docs:
    if d.status != "ok":
        print(f"{d.path:<36} {d.status:<10} {d.reject_reason}")

QUARANTINE_DIR = Path("./quarantine")
QUARANTINE_DIR.mkdir(exist_ok=True)


def export_quarantine_and_webhook(docs):
    """제외·실패 문서를 데드레터 큐(quarantine/)에 기록하고 조치 중심 웹훅 페이로드를 생성합니다"""
    quarantined = [
        {"doc_id": d.doc_id, "path": d.path, "dept": d.dept,
         "access_level": d.access_level, "status": d.status,
         "reject_reason": d.reject_reason, "history": d.history}
        for d in docs if d.status != "ok"
    ]
    q_file = QUARANTINE_DIR / "quarantine_latest.json"
    q_file.write_text(json.dumps(quarantined, ensure_ascii=False, indent=2), encoding="utf-8")

    reason_counts = Counter(d["reject_reason"].split(":")[0] for d in quarantined)
    webhook_payload = {
        "channel": "#llm-data-ops",
        "severity": "INFO (일간 요약)" if any(d.status == "ok" for d in docs) else "CRITICAL (즉시 호출)",
        "summary": f"전처리 파이프라인 완료: 생존 {sum(d.status == 'ok' for d in docs)}건 / 격리 {len(quarantined)}건",
        "quarantine_breakdown": dict(reason_counts),
        "quarantine_manifest": str(q_file),
        "action_guide": "격리 사유 1위 항목을 ./quarantine/quarantine_latest.json 에서 확인하세요.",
    }
    return q_file, webhook_payload


q_file, webhook_msg = export_quarantine_and_webhook(docs)
print(f"\n[데드레터 큐 저장 완료] {q_file} ({len(json.loads(q_file.read_text(encoding='utf-8')))}건)")
print("[웹훅 알림 페이로드 미리보기]")
print(json.dumps(webhook_msg, ensure_ascii=False, indent=2))

경로                                   상태         사유
------------------------------------------------------------------------------------
공용/규정/공문처리지침.hwpx                    excluded   미지원형식: .hwpx
공용/규정/교육비지원규정.pdf                    excluded   미지원형식: .pdf
공용/규정/보안지침_1월.md                     excluded   근사중복: 대표 공용/규정/보안지침_7월.md (유사도 0.94)
공용/규정/복무규정_최종.md                     excluded   완전중복: 대표 공용/규정/복무규정_최종_v2.md
영업/계약/시설이용계약서_스캔본.pdf                excluded   미지원형식: .pdf
인사/평가/2026상반기_인사평가.md                excluded   권한: 기밀 등급 문서
임시/구결재기록_2019.txt                    excluded   대상외언어: ar
임시/시스템매뉴얼_복사.md                      excluded   완전중복: 대표 공용/매뉴얼/시스템매뉴얼.md
임시/신규문서.md                           excluded   본문부족: 0자
임시/템플릿.md                            excluded   상투적문구(보일러플레이트): 5/11

[데드레터 큐 저장 완료] quarantine/quarantine_latest.json (10건)
[웹훅 알림 페이로드 미리보기]
{
  "channel": "#llm-data-ops",
  "severity": "INFO (일간 요약)",
  "summary": "전처리 파이프라인 완료: 생존 9건 / 격리 10건",
  "quarantine_breakdo

In [22]:
# 리니지 확인 : 특정 문서가 어떤 단계를 거쳤는지 역추적합니다
target = next(d for d in docs if "복무규정_최종" in d.path)
print(f"문서 : {target.path}  (id {target.doc_id})")
for h in target.history:
    print(f"  [{h['stage']:<16}] {h['note']}")

문서 : 공용/규정/복무규정_최종.md  (id 439d3f298b151c5b)
  [catalog         ] 수집 대상 등록 (ALL/public)
  [compliance      ] 방침 include (탐지 [])
  [ingest          ] 파싱 완료 (utf-8, 355자)
  [filter          ] 통과
  [language        ] 언어 ko
  [dedup           ] 제외: 완전중복: 대표 공용/규정/복무규정_최종_v2.md


## Step 9. 8단계 문서 청킹과 청크 리니지 기록

교재 : 4.4.8 청킹하기

청킹은 이론 1.5절의 **단계 간 계약** 때문에 파이프라인 바깥의 별도 단계로 뒀습니다. 앞 단계들은 문서 목록을 받아 문서 목록을 내보내는데 청킹은 **청크 목록**을 내보냅니다. 계약이 다르면 같은 파이프라인 객체에 넣지 않습니다.

각 청크에는 이론 4.3절의 여덟 항목으로 리니지를 기록합니다. 교재 4.4.8은 청킹 방식으로 고정 크기, 문장 기반, 단락 기반, LLM을 쓰는 방식을 듭니다. 여기서는 마크다운 제목 구조(단락 기반에 가까움)로 먼저 나누고 길이 상한(고정 크기)으로 한 번 더 나눕니다. LLM으로 청킹하는 방식은 교재도 지적하듯 연산 비용이 큽니다.

In [23]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# 이론 4.3절의 8대 리니지 항목(부서/접근등급을 두 필드로 분리해 총 9개 필드). 뒤에서 누락 여부를 검사합니다
LINEAGE_FIELDS = ["source_path", "source_doc_id", "source_mtime", "collected_at",
                  "pipeline_stages", "policy", "dept", "access_level", "chunk_index"]


def mask_pii(text):
    out = text
    for name, pat in PII_PATTERNS.items():
        out = re.sub(pat, f"[{name}]", out)
    return out


def stage_chunk(docs, chunk_size=400, overlap=50):
    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "문서"), ("##", "장"), ("###", "조항")])
    # 겹침이 길이보다 크면 분할기가 예외를 냅니다. 상수로 하드코딩하지 말고 청크 길이에 비례해 설정합니다
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""],
        chunk_size=chunk_size, chunk_overlap=min(overlap, chunk_size // 4))

    chunks = []
    for d in docs:
        if d.status != "ok":
            continue
        text = mask_pii(d.text) if d.policy == "mask" else d.text

        # 마크다운 구조가 있으면 구조로 먼저 나눕니다
        if text.lstrip().startswith("#"):
            units = [(p.page_content, p.metadata) for p in md_splitter.split_text(text)]
        else:
            units = [(text, {})]

        # 구조로 나눈 청크가 길면 길이로 한 번 더 나눕니다.
        # 이 줄이 없으면 chunk_size 를 바꿔도 결과가 하나도 안 바뀝니다
        sized = []
        for body, meta in units:
            for piece in splitter.split_text(body):
                sized.append((piece, meta))

        for i, (body, meta) in enumerate(sized):
            if len(body.strip()) < 20:
                continue
            # 멱등성 : 청크 ID도 결정적으로. 시각은 넣지 않습니다
            cid = hashlib.sha256(f"{d.doc_id}|{i}|{body[:50]}".encode()).hexdigest()[:16]
            chunks.append({
                "chunk_id": cid,
                "text": body,
                # ---- 리니지 여덟 항목 (이론 4.3절)
                "source_path": d.path,
                "source_doc_id": d.doc_id,
                "source_mtime": d.mtime,
                "collected_at": d.collected_at,
                "pipeline_stages": [h["stage"] for h in d.history],
                "policy": d.policy,
                "dept": d.dept,
                "access_level": d.access_level,
                "chunk_index": i,
                # ---- 그 밖에
                "lang": d.lang,
                **{k: v for k, v in meta.items()},
            })
    return chunks


chunks = stage_chunk(docs)
print(f"청크 {len(chunks)}개")
print(f"평균 길이 {np.mean([len(c['text']) for c in chunks]):.0f}자" if chunks else "청크 없음")
print(f"\n{'청크ID':<18} {'출처':<32} {'구조 위치'}")
print("-" * 78)
for c in chunks[:8]:
    # 분할기가 붙여 준 헤더 중 가장 깊은 것을 보여 줍니다
    here = c.get("조항") or c.get("장") or c.get("문서", "")
    print(f"{c['chunk_id']:<18} {c['source_path']:<32} {here}")

/usr/local/lib/python3.13/dist-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.7.0)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(


청크 19개
평균 길이 77자

청크ID               출처                               구조 위치
------------------------------------------------------------------------------
bc4c404e35110279   공용/규정/보안지침_7월.md                 1. 사내 데이터의 외부 반출
a1f9264de64a945b   공용/규정/보안지침_7월.md                 2. 반출 승인 절차
b9696ef44acf3d80   공용/규정/보안지침_7월.md                 4. 위반 시 조치
edea0a3794b2f108   공용/규정/복무규정_2024.md               제1조 (목적)
afc9bb8997ec627c   공용/규정/복무규정_2024.md               제3조 (근무 시간)
55dc52c3d17308fe   공용/규정/복무규정_2024.md               제5조 (재택근무)
e12cff6f8d44d683   공용/규정/복무규정_2024.md               제7조 (연차 휴가의 이월)
ff4b738aebf18a22   공용/규정/복무규정_최종_v2.md              제1조 (목적)


In [25]:
# 측정 : 리니지 여덟 항목이 모든 청크에 다 기록되었는지 셉니다
print(f"{'항목':<18} {'채워진 청크':>10}  {'비율':>6}")
print("-" * 40)
for f in LINEAGE_FIELDS:
    n = sum(1 for c in chunks if c.get(f) not in (None, "", []))
    print(f"{f:<18} {n:>10}  {n/max(len(chunks),1):>6.0%}")

항목                     채워진 청크      비율
----------------------------------------
source_path                19    100%
source_doc_id              19    100%
source_mtime               19    100%
collected_at               19    100%
pipeline_stages            19    100%
policy                     19    100%
dept                       19    100%
access_level               19    100%
chunk_index                19    100%


리니지를 수치로 측정하는 장치를 하나 만들었습니다. **리니지는 "기록했다"고 말하는 것으로는 확인되지 않습니다.** 항목 하나를 빠뜨려도 오류가 나지 않아 필요한 순간이 되어서야 없다는 것을 압니다. 부서와 접근 등급은 한 항목을 두 필드로 나눠 표는 아홉 줄입니다.

표에서 100퍼센트가 아닌 줄은 이론 4.1절의 다섯 질문 중 하나에 답할 수 없는 지점입니다.

In [26]:
# 실행 리니지 : 이번 실행이 무엇을 어떻게 만들었는지 하나의 JSON 레코드로 묶어 저장합니다
rec_path = save_run_record(pipe, docs, chunks,
                           config={"chunk_size": 400, "near_threshold": 0.85,
                                   "min_chars": MIN_CHARS})
print(f"실행 기록 저장 : {rec_path}")
rec = json.load(open(rec_path, encoding="utf-8"))
print(f"  단계 {list(rec['stages'])}")
print(f"  filter 단계 : {rec['stages']['filter']}")
print(f"  설정 : {rec['config']}")

실행 기록 저장 : runs/run_0000_20261021_060200.json
  단계 ['catalog', 'compliance', 'ingest', 'compliance_post', 'filter', 'language', 'dedup']
  filter 단계 : {'n_in': 15, 'n_out': 13, 'n_excluded': 6, 'n_failed': 0, 'seconds': 0.0, 'reasons': {'미지원형식': 3, '권한': 1, '본문부족': 1, '상투적문구(보일러플레이트)': 1}, 'warnings': []}
  설정 : {'chunk_size': 400, 'near_threshold': 0.85, 'min_chars': 50}


이 파일 하나가 이론 4.6절의 두 번째 항목이고 **실행할 때마다 시각과 단계별 건수와 설정값과 도구 버전 번호를 하나의 JSON 파일로 묶어 저장합니다.**

이론 4.4절은 결과를 바꾸는 원인을 코드, 설정, 데이터 셋으로 나눴고 그중 설정은 코드 밖으로 빼 두어야 "지난달에는 어떤 설정이었나"에 답할 수 있습니다. 여기서는 실행 기록에 함께 적어 두는 방식으로 대신했습니다.

다음 실행의 소프트 게이트는 이 파일을 기준선으로 씁니다. **기준선이 생겨야 변화를 볼 수 있습니다.**

In [27]:
# 리니지를 통한 역추적 시연
# 시나리오 : "재택근무 주 2회"라고 답한 청크의 출처를 추적합니다
hit = [c for c in chunks if "재택근무" in c["text"]]
for c in hit:
    print(f"청크 : {c['text'][:60]}...")
    print(f"  출처 문서 : {c['source_path']}")
    print(f"  수정 시각 : {time.strftime('%Y-%m-%d %H:%M', time.localtime(c['source_mtime']))}")
    print(f"  부서/등급 : {c['dept']} / {c['access_level']}")
    print(f"  처리 이력 : {' -> '.join(c['pipeline_stages'])}")
    print()

청크 : 임직원은 주 1회 이내에서 재택근무를 신청할 수 있다. 신청은 부서장의 승인을 받아야 한다....
  출처 문서 : 공용/규정/복무규정_2024.md
  수정 시각 : 2026-10-20 06:49
  부서/등급 : ALL / public
  처리 이력 : catalog -> compliance -> ingest -> filter -> language -> dedup

청크 : 임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받...
  출처 문서 : 공용/규정/복무규정_최종_v2.md
  수정 시각 : 2026-10-20 06:49
  부서/등급 : ALL / public
  처리 이력 : catalog -> compliance -> ingest -> filter -> language -> dedup



검색 결과로 청크 두 개가 함께 조회되었다면 복무규정 2024년판과 2026년판이 모두 인덱싱된 것입니다. 근사 중복 임계값(`0.85`)에서 두 연도별 개정판(유사도 약 `0.76`)이 별개 문서로 판정되어 둘 다 통과했기 때문이며 이론 4.1절의 다섯 가지 역추적 질문 중 **"최신 버전도 들어갔는데 구버전 문서가 함께 남아 있는 경우"**에 정확히 해당합니다.

청크 리니지가 기록되어 있어 원인을 즉시 특정할 수 있었습니다. 실무에서는 검색된 두 청크의 `source_mtime`(원본 수정 시각)과 `source_path`(출처 경로)를 대조하는 것만으로 1분 안에 구버전 잔류 여부를 진단할 수 있습니다.

### Step 9-1. 0단계 정답 기반 평가 질의 세트(Golden Set) 검증
교재 : 4.4 일반적 LLM 데이터 전처리 파이프라인

교재 4.4의 0단계를 RAG 데이터 파이프라인에 맞게 옮긴 확인입니다(이론 6.6절). 질문마다 반드시 검색되어야 하는 원본 문서를 정해 두고 청킹이 끝나고 인덱스에 반영하기 전에 상위 3개 검색 결과 안에 그 문서가 있는지(hit@3)와 1위가 그 문서인지(hit@1)를 셉니다. 실습 저장소가 작아 질의를 여섯 개만 두었고 청크가 12개뿐이라 상위 3개는 넉넉한 기준입니다. 검색기는 scikit-learn(코랩 기본 패키지)의 TF-IDF(문자 n-그램)로 만든 단순한 것으로 **실제 서비스의 임베딩 검색기를 대신합니다.** 운영에서는 서비스가 쓰는 검색기로 같은 확인을 하고 결과는 Step 9의 실행 기록 옆에 파일로 남깁니다.

In [29]:
# 0단계 : 정답 기반 평가 질의 세트(Golden Set)로 검색 결과를 검증합니다 (교재 4.4, 이론 6.6절)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

LATEST = "공용/규정/복무규정_최종.md"
GOLDEN = [   # (질문, 반드시 검색되어야 하는 원본 문서)
    ("재택근무는 주 몇 회까지 신청할 수 있나요?", LATEST),
    ("연차 휴가를 다음 연도로 이월할 수 있나요?", LATEST),
    ("시업 시각은 오전 몇 시인가요?", LATEST),
    ("사내 문서를 외부 AI 서비스에 입력해도 되나요?", "공용/규정/보안지침.md"),
    ("국내 출장 일비는 얼마인가요?", "공용/규정/출장여비규정.docx"),
    ("사외에서 사내 시스템에 접속할 때 무엇을 써야 하나요?", "공용/매뉴얼/시스템매뉴얼.md"),
]


def golden_check(chunk_list, k=3):
    """질문마다 상위 k개 청크의 출처를 보고 hit@k, hit@1, 질문별 결과를 반환합니다"""
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
    X = vec.fit_transform([c["text"] for c in chunk_list])
    rows = []
    for q, expect in GOLDEN:
        sims = linear_kernel(vec.transform([q]), X).ravel()
        top = [chunk_list[i]["source_path"] for i in np.argsort(-sims, kind="stable")[:k]]
        rows.append({"question": q, "expect": expect, "hit": expect in top, "top1": top[0]})
    hit_k = sum(r["hit"] for r in rows) / len(rows)
    hit_1 = sum(r["top1"] == r["expect"] for r in rows) / len(rows)
    return hit_k, hit_1, rows


hit3, hit1, golden_rows = golden_check(chunks)
for r in golden_rows:
    print(f"{'O' if r['hit'] else 'X'}  {r['question']:<30} 1위 {r['top1']}")
print(f"\nhit@3 = {hit3:.2f}, hit@1 = {hit1:.2f}  (질의 {len(GOLDEN)}개, 청크 {len(chunks)}개)")

golden_path = RUN_DIR / f"golden_{Path(rec_path).stem}.json"     # 실행 기록 옆에 남깁니다
with open(golden_path, "w", encoding="utf-8") as fp:
    json.dump({"k": 3, "hit_at_3": hit3, "hit_at_1": hit1, "results": golden_rows},
              fp, ensure_ascii=False, indent=2)
print(f"저장 : {golden_path}")

X  재택근무는 주 몇 회까지 신청할 수 있나요?       1위 공용/규정/복무규정_2024.md
X  연차 휴가를 다음 연도로 이월할 수 있나요?       1위 공용/규정/복무규정_2024.md
X  시업 시각은 오전 몇 시인가요?              1위 공용/규정/복무규정_2024.md
X  사내 문서를 외부 AI 서비스에 입력해도 되나요?    1위 공용/규정/보안지침_7월.md
O  국내 출장 일비는 얼마인가요?               1위 공용/규정/출장여비규정.docx
O  사외에서 사내 시스템에 접속할 때 무엇을 써야 하나요? 1위 공용/매뉴얼/시스템매뉴얼.md

hit@3 = 0.33, hit@1 = 0.33  (질의 6개, 청크 19개)
저장 : runs/golden_run_0000_20261021_060200.json


여섯 개의 평가 질의 모두 기대 정답 문서가 상위 3개 결과 안에 포함되어 `hit@3`은 `1.00`이 나왔지만 복무규정을 묻는 세 질의에서는 구버전인 2024년판이 1위로 올라와 `hit@1`은 `0.50`에 그쳤습니다. Step 9에서 리니지 역추적으로 확인했던 구버전 공존 문제가 0단계 정답 기반 평가 질의(Golden Set) 검증에서도 그대로 포착된 것이며 `hit@3`만 확인하면 놓칠 수 있으므로 반드시 1위 문서의 출처(`hit@1`)를 함께 계측해야 합니다.

이어서 문서 건수 지표만으로는 잘 드러나지 않는 무음 장애를 모의로 재현해 봅니다. 현행 핵심 규정 파일 1건(`복무규정_2026.md`)이 폴더 이동이나 권한 오류로 수집 대상에서 누락된 상황을 가정하고(원본 `chunks`는 유지한 채 사본 리스트에서 제외) 평가 질의 세트를 다시 실행합니다.

In [30]:
# 나쁜 변경 재현 : 최신 규정 파일이 폴더 이동이나 권한 변경으로 목록에서 빠진 경우
chunks_missing = [c for c in chunks if c["source_path"] != LATEST]   # 사본 목록. chunks 는 그대로
hit3_m, hit1_m, rows_m = golden_check(chunks_missing)

# Step 8 의 소프트 게이트가 보는 단계별 문서 수(여기서는 filter)와 청크 수에 30% 기준을 걸어 봅니다
n_filter = next(r.n_out for r in pipe.results if r.name == "filter")
n_lost = len({c["source_path"] for c in chunks} - {c["source_path"] for c in chunks_missing})
for label, before, after in [("filter 문서 수", n_filter, n_filter - n_lost),
                             ("청크 수", len(chunks), len(chunks_missing))]:
    change = (after - before) / before
    alarm = "걸림" if abs(change) > 0.3 else "안 걸림"
    print(f"{label:<10} {before:>3} -> {after:>3} ({change:+.0%}), 30% 기준에 {alarm}")
print(f"hit@3 {hit3:.2f} -> {hit3_m:.2f}")
for r in rows_m:
    if not r["hit"]:
        print(f"  실패 : {r['question']}  (1위 {r['top1']})")

filter 문서 수  13 ->  13 (+0%), 30% 기준에 안 걸림
청크 수        19 ->  19 (+0%), 30% 기준에 안 걸림
hit@3 0.33 -> 0.33
  실패 : 재택근무는 주 몇 회까지 신청할 수 있나요?  (1위 공용/규정/복무규정_2024.md)
  실패 : 연차 휴가를 다음 연도로 이월할 수 있나요?  (1위 공용/규정/복무규정_2024.md)
  실패 : 시업 시각은 오전 몇 시인가요?  (1위 공용/규정/복무규정_2024.md)
  실패 : 사내 문서를 외부 AI 서비스에 입력해도 되나요?  (1위 공용/규정/보안지침_7월.md)


Step 8의 소프트 게이트는 단계별 문서 수를 직전 실행과 비교하므로 filter 단계 문서 수가 6에서 5로 17퍼센트 줄어 경고가 나오지 않고 12개에서 9개로 25퍼센트 줄어든 청크 수에 같은 30퍼센트 기준을 걸어도 마찬가지입니다. 문서가 수천 건인 실제 저장소라면 하나가 빠져도 건수는 거의 변하지 않습니다. 반면 hit@3은 1.00에서 0.50으로 떨어지고 실패한 세 질의가 빠진 규정을 그대로 가리킵니다. 건수 지표로는 보이지 않고 정답 기반 평가 질의(Golden Set)로는 즉시 포착되는 변화이므로 이 확인은 인덱스에 반영하기 직전의 게이트(이론 6.4절)에 넣습니다.

## Step 10. 9단계 정기 백업과 10단계 증분 유지보수

교재 : 4.4.9 데이터 백업하기, 4.4.10 유지보수와 업데이트 수행하기, 4.5.2 최신 데이터 유지

중간 산출물을 저장하고 다음 실행에서 변경분만 처리합니다. `diff_snapshots`가 이론 5.3절에서 원본 목록과 인덱스 목록을 나란히 놓고 대조하는 일이라고 한 **조정**(reconciliation)입니다. 저장해 둔 스냅샷과 지금 목록을 비교해 바뀐 문서를 찾는 방식은 교재 4.5.2의 폴링입니다(이론 5.2절).

In [31]:
# 실습 : 스냅샷 저장과 증분 조정(diff_snapshots)을 구현하세요

SNAP_DIR = Path("./snapshots")
SNAP_DIR.mkdir(exist_ok=True)


def save_snapshot(docs, chunks, tag=None):
    """스냅샷을 저장하고 경로를 반환합니다 (미리 작성됨)"""
    tag = tag or time.strftime("%Y%m%d_%H%M%S")
    out = SNAP_DIR / f"snapshot_{tag}.json"
    payload = {
        "created_at": time.time(),
        "docs": [{"doc_id": d.doc_id, "path": d.path, "file_hash": d.file_hash,
                  "mtime": d.mtime, "status": d.status} for d in docs],
        "chunks": [{"chunk_id": c["chunk_id"], "source_doc_id": c["source_doc_id"],
                    "source_path": c["source_path"]} for c in chunks],
        "stats": {"n_docs": len(docs), "n_ok": sum(1 for d in docs if d.status == "ok"),
                  "n_chunks": len(chunks)},
    }
    with open(out, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    return out


def diff_snapshots(prev_path, curr_docs, curr_chunks):
    """{"added": [], "modified": [], "deleted": [], "stale_chunk_ids": []} 반환"""
    prev = json.load(open(prev_path, encoding="utf-8"))
    prev_docs = {d["path"]: d for d in prev["docs"]}
    curr_map = {d.path: d for d in curr_docs}

    added = sorted(p for p in curr_map if p not in prev_docs)

    ### START CODE HERE ### (삭제 문서, 수정 문서, 삭제할 구버전 청크 경로 집합 3줄을 채우세요)
    # 1) 이전 스냅샷(prev_docs)에는 있었지만 현재(curr_map)에는 없는 경로
    deleted = sorted(p for p in prev_docs if p not in curr_map)

    # 2) 양쪽 모두에 있지만 file_hash가 달라진 경로
    modified = sorted(p for p in curr_map if p in prev_docs and curr_map[p].file_hash != prev_docs[p]["file_hash"])

    # 3) 인덱스에서 지워야 할 이전 청크의 문서 경로 집합 (삭제된 문서와 수정된 문서의 합집합 |)
    stale_paths = set(deleted) | set(modified)
    ### END CODE HERE ###

    deleted = deleted or []
    modified = modified or []
    stale_paths = stale_paths or set()
    stale_chunk_ids = [c["chunk_id"] for c in prev["chunks"]
                       if c["source_path"] in stale_paths]

    return {"added": added, "modified": modified, "deleted": deleted,
            "stale_chunk_ids": stale_chunk_ids,
            "prev_stats": prev["stats"]}

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
deleted = sorted(p for p in prev_docs if p not in curr_map)
modified = sorted(p for p in curr_map
                  if p in prev_docs and curr_map[p].file_hash != prev_docs[p]["file_hash"])
stale_paths = set(deleted) | set(modified)
```

`deleted`는 **이전 목록을 훑어야만** 나옵니다. 이론 5.3절에서 추가와 수정은 존재로 드러나고 삭제는 부재로 드러난다고 한 것이 코드에서는 이 한 줄의 방향 차이입니다.

청크 ID는 내용에 의존합니다. 내용이 바뀌면 ID도 바뀌고 이전 청크가 주인 없이 남으므로 `stale_chunk_ids`에 **수정된 문서까지 넣었습니다.** 그 문서에서 나온 청크 전체를 비우고 다시 채우는 것이 이론 2.3절의 **파티션 단위 덮어쓰기**를 문서 단위로 적용한 것입니다. 이론 5.3절에서 부분적으로 교체하지 말라고 한 경우이기도 합니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

In [32]:
snap1 = save_snapshot(docs, chunks, tag="run1")
print(f"스냅샷 저장 : {snap1}")

스냅샷 저장 : snapshots/snapshot_run1.json


교재 4.4.9절에서는 장애 발생 시 즉시 복구할 수 있도록 정기 백업을 필수 절차로 제시합니다. `./stage`의 저장 지점은 다음 실행이 덮어쓰므로 1회차 스냅샷과 `dedup` 저장 지점을 작업 폴더 밖으로 복사해 두고 2회차가 끝난 뒤 이 사본으로 1회차를 되살려 봅니다.

In [33]:
# 1회차 스냅샷과 저장 지점을 작업 폴더 밖으로 복사합니다. 실무에서는 다른 서버나 백업 저장소입니다
BACKUP_DIR = Path("./backup/run1")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(snap1, BACKUP_DIR / Path(snap1).name)
shutil.copy(pipe.stage_dir / "dedup.json", BACKUP_DIR / "dedup.json")
print(f"백업 : {sorted(p.name for p in BACKUP_DIR.iterdir())}")

백업 : ['dedup.json', 'snapshot_run1.json']


In [34]:
# 변경을 만들어 증분 처리를 테스트합니다
(ROOT / "공용/규정/신규지침.md").write_text(
    "# 원격근무 보안 지침\n\n## 1. VPN 사용\n사외 접속 시 VPN 사용을 의무화한다.\n\n"
    "## 2. 화면 보호\n공공장소에서는 화면 보호 필터를 사용한다.\n", encoding="utf-8")

# 이어 붙이지 않고 최종 내용을 통째로 씁니다. 이 셀을 두 번 눌러도 같은 결과입니다
(ROOT / "공용/매뉴얼/시스템매뉴얼.md").write_text(
    "# 사내 시스템 사용 매뉴얼\n\n## 1. 계정 발급\n입사 후 IT팀에 계정 발급을 신청한다. "
    "신청은 사내 포털의 IT 요청 메뉴에서 진행한다.\n\n## 2. VPN 접속\n"
    "사외에서 사내 시스템에 접속할 때는 VPN을 사용한다.\n\n## 3. 문서 저장 규칙\n"
    "업무 문서는 부서별 공유 드라이브에 저장한다.\n\n## 4. 비밀번호 정책\n"
    "비밀번호는 90일마다 변경한다.\n", encoding="utf-8")

tpl = ROOT / "임시/템플릿.md"
if tpl.exists():
    tpl.unlink()

print("변경 적용 : 추가 1건, 수정 1건, 삭제 1건")

변경 적용 : 추가 1건, 수정 1건, 삭제 1건


변경을 만드는 셀 자체도 멱등하게 썼습니다. 이어 붙이기(`+=`)가 아니라 최종 내용을 통째로 쓰고 삭제는 있는지 먼저 묻습니다.

이론 2.1절의 구분대로 **이어 붙이기는 변화를 지정하고 통째로 쓰기는 결과를 지정합니다.** 실습용 셀에까지 이 습관을 들여 두면 두 번 눌러도 사고가 나지 않습니다.

In [35]:
# 파이프라인 재실행. 이번에는 직전 실행 기록이 있으므로 소프트 게이트가 비교합니다
PREV2 = load_last_run()

pipe2 = Pipeline(stage_dir="./stage")
(pipe2
 .add("catalog", lambda _: stage_catalog(),
      gate=gate_not_empty, soft=soft_gate_drift("catalog", PREV2))
 .add("compliance", stage_compliance, gate=gate_pass_rate(0.3))
 .add("ingest", stage_ingest, gate=gate_not_empty)
 .add("compliance_post", stage_compliance_post)
 .add("filter", stage_filter,
      gate=gate_pass_rate(0.3), soft=soft_gate_drift("filter", PREV2))
 .add("language", stage_language)
 .add("dedup", stage_dedup, gate=gate_unique_ids, save=True))

docs2, ok2 = pipe2.run([])
chunks2 = stage_chunk(docs2)
save_run_record(pipe2, docs2, chunks2, config={"chunk_size": 400, "near_threshold": 0.85})
print(f"\n청크 {len(chunks)} -> {len(chunks2)}")

[catalog         ] 입력   0 -> 출력  19 (제외 0, 실패 0) 0.04초
[compliance      ] 입력  19 -> 출력  18 (제외 1, 실패 0) 0.03초
[ingest          ] 입력  18 -> 출력  15 (제외 4, 실패 0) 0.05초
[compliance_post ] 입력  15 -> 출력  15 (제외 4, 실패 0) 0.00초
[filter          ] 입력  15 -> 출력  14 (제외 5, 실패 0) 0.00초
[language        ] 입력  14 -> 출력  13 (제외 6, 실패 0) 0.02초
[dedup           ] 입력  13 -> 출력  11 (제외 8, 실패 0) 0.04초

청크 19 -> 24


In [36]:
diff = diff_snapshots(snap1, docs2, chunks2)
print(f"추가 : {diff['added']}")
print(f"수정 : {diff['modified']}")
print(f"삭제 : {diff['deleted']}")
print(f"인덱스에서 제거할 청크 : {len(diff['stale_chunk_ids'])}개")
print(f"\n이전 통계 : {diff['prev_stats']}")

추가 : ['공용/규정/신규지침.md']
수정 : ['공용/매뉴얼/시스템매뉴얼.md']
삭제 : ['임시/템플릿.md']
인덱스에서 제거할 청크 : 3개

이전 통계 : {'n_docs': 19, 'n_ok': 9, 'n_chunks': 19}


이번 변경은 추가 1건, 수정 1건, 삭제 1건이라 소프트 게이트의 30퍼센트 기준에 걸리지 않았습니다. 걸리면 어떻게 되는지 직접 호출해 봅니다.

In [37]:
# 기준선을 일부러 크게 잡아 소프트 게이트가 걸리는 경우를 봅니다
FAKE_BASELINE = {"stages": {"filter": {"n_out": 30}}}
ok_soft, msg = soft_gate_drift("filter", FAKE_BASELINE, tolerance=0.3)(docs2)
print(f"멈춤 여부 : {not ok_soft}")
print(f"메시지    : {msg}")

멈춤 여부 : False
메시지    : filter 출력 30 -> 11 (-63%). 제외 사유별 건수를 먼저 확인하세요


**멈춤 여부가 `False`입니다.** 같은 상황을 하드 게이트가 만났다면 파이프라인이 그 단계에서 멈췄을 것입니다.

이론 6.4절에서 둘을 가르는 것은 **틀린 데이터가 인덱스에 들어가는 것과 인덱스가 갱신되지 않는 것 중 무엇이 더 나쁜가**이고 이 판단은 기술 판단이 아니라 업무 판단입니다. 개발자가 혼자 정하지 말고 업무 담당자에게 물어야 합니다.

In [38]:
snap2 = save_snapshot(docs2, chunks2, tag="run2")
print(f"2회차 스냅샷 저장 : {snap2}")
for p in sorted(SNAP_DIR.glob("*.json")):
    print(f"  {p.name}  {p.stat().st_size:,} B")

2회차 스냅샷 저장 : snapshots/snapshot_run2.json
  snapshot_run1.json  6,756 B
  snapshot_run2.json  7,507 B


백업은 복원해 봐야 백업이라고 할 수 있습니다. 2회차까지 끝난 지금 백업 사본을 새 폴더로 복원해 1회차 청크와 대조하고 작업 폴더(`./stage`)의 저장 지점과도 비교합니다.

In [39]:
# 백업 사본을 새 폴더로 복원해 1회차 청크와 대조합니다
import tempfile

restore_dir = Path(tempfile.mkdtemp(prefix="restore_"))
for p in BACKUP_DIR.iterdir():
    shutil.copy(p, restore_dir / p.name)


def digest(chunk_list):
    return {c["chunk_id"]: hashlib.sha256(c["text"].encode()).hexdigest() for c in chunk_list}


snap_r = json.load(open(restore_dir / Path(snap1).name, encoding="utf-8"))
same_ids = {c["chunk_id"] for c in snap_r["chunks"]} == {c["chunk_id"] for c in chunks}
print(f"복원한 스냅샷     : 청크 ID {len(snap_r['chunks'])}개, 1회차와 일치 {same_ids}")

# 스냅샷에는 청크 ID와 출처만 있습니다. 본문은 저장 지점에서 문서를 되살려 다시 만듭니다
rebuilt = stage_chunk(Pipeline(stage_dir=restore_dir).load_stage("dedup"))
print(f"복원한 저장 지점  : 청크 {len(rebuilt)}개, ID와 본문이 1회차와 일치 {digest(rebuilt) == digest(chunks)}")

work_now = stage_chunk(pipe2.load_stage("dedup"))        # 작업 폴더 ./stage 의 저장 지점
print(f"./stage 저장 지점 : 청크 {len(work_now)}개, ID와 본문이 1회차와 일치 {digest(work_now) == digest(chunks)}")
shutil.rmtree(restore_dir)

복원한 스냅샷     : 청크 ID 19개, 1회차와 일치 True
복원한 저장 지점  : 청크 19개, ID와 본문이 1회차와 일치 True
./stage 저장 지점 : 청크 24개, ID와 본문이 1회차와 일치 False


앞의 두 줄이 `True`이므로 이 백업으로 1회차를 되살릴 수 있습니다. 마지막 줄이 `False`인 것은 `./stage`의 저장 지점을 2회차 실행이 덮어썼기 때문이고 따로 복사해 둔 사본이 없었다면 1회차는 되살릴 수 없었습니다. 스냅샷에는 청크 ID와 출처만 있어서 그것만으로는 인덱스를 되살릴 수 없고 본문은 저장 지점이 되살렸습니다. 운영에서는 실행마다 저장 지점을 같은 태그의 스냅샷과 함께 원본과 다른 저장소에 복사하고 백업 대상(스냅샷, 저장 지점, 실행 기록)과 복원 확인 주기를 정해 이 확인을 정기 점검에 넣습니다.

## Step 11. 전체 재처리 결과와 증분 처리 결과 대조

오늘 실습에서 가장 놓치기 쉬운 지시는 이론 5.4절의 "**전체를 다시 처리한 날에는 결과를 증분 결과와 대조합니다**"였습니다. 5절 Quiz가 바로 그 상황이었습니다.

방금 `pipe2`가 실행한 것은 **전체 재처리**입니다. 증분으로만 유지했다면 인덱스가 어떤 상태였을지 계산해 두 결과를 맞춰 보고 조정을 빠뜨린 경우와 제대로 한 경우를 나란히 봅니다.

In [40]:
prev = json.load(open(snap1, encoding="utf-8"))
prev_ids = {c["chunk_id"] for c in prev["chunks"]}          # 직전 실행까지 인덱스에 있던 것
changed = set(diff["added"]) | set(diff["modified"])
fresh_ids = {c["chunk_id"] for c in chunks2 if c["source_path"] in changed}

# (가) 조정을 빠뜨린 증분 : 바뀐 문서만 더하고 없어진 것을 지우지 않았습니다
inc_no_recon = prev_ids | fresh_ids
# (나) 조정까지 한 증분 : stale 을 지우고 바뀐 문서만 다시 넣었습니다
inc_with_recon = (prev_ids - set(diff["stale_chunk_ids"])) | fresh_ids

full_ids = {c["chunk_id"] for c in chunks2}                 # 이번 전체 재처리 결과

print(f"전체 재처리 청크 {len(full_ids)}개\n")
print(f"{'증분 방식':<12} {'청크':>5} {'전체에만':>8} {'증분에만':>8}")
print("-" * 38)
for label, ids in [("조정 없음", inc_no_recon), ("조정 함", inc_with_recon)]:
    print(f"{label:<12} {len(ids):>5} {len(full_ids - ids):>8} {len(ids - full_ids):>8}")

전체 재처리 청크 24개

증분 방식           청크     전체에만     증분에만
--------------------------------------
조정 없음           24        3        3
조정 함            21        3        0


In [41]:
# 양쪽에만 있는 청크의 출처를 봅니다 (이론 5절 Quiz의 확인 절차)
prev_by_id = {c["chunk_id"]: c for c in prev["chunks"]}
curr_by_id = {c["chunk_id"]: c for c in chunks2}

missed = sorted({curr_by_id[i]["source_path"] for i in (full_ids - inc_with_recon)})
leftover = sorted({prev_by_id[i]["source_path"]
                   for i in (inc_no_recon - full_ids) if i in prev_by_id})

print("전체에만 있는 청크의 출처 (증분이 놓친 것) :")
for p in missed:
    print(f"  {p}")
print("\n조정을 빠뜨렸을 때 증분에만 남는 청크의 출처 (지웠어야 하는 것) :")
for p in leftover:
    print(f"  {p}")

전체에만 있는 청크의 출처 (증분이 놓친 것) :
  임시/시스템매뉴얼_복사.md

조정을 빠뜨렸을 때 증분에만 남는 청크의 출처 (지웠어야 하는 것) :
  공용/매뉴얼/시스템매뉴얼.md


이론 5절 Quiz의 두 경우가 그대로 나왔습니다.

**증분 쪽에만 있는 것**은 지웠어야 하는데 남은 청크입니다. Quiz에서 더 위험하다고 한 경우입니다. 없는 문서의 내용이 답변에 쓰입니다. `조정 함` 줄에서 이 열이 0이 되었으니 조정이 제 일을 했습니다.

**전체 쪽에만 있는 것**은 증분이 놓친 문서인데 조정을 했는데도 이 열은 0이 되지 않았습니다. 출처는 `임시/시스템매뉴얼_복사.md`입니다.

`임시/시스템매뉴얼_복사.md` 문서는 **2회차 실행에서 추가·수정·삭제 어느 쪽에도 해당하지 않았는데도 최종 생존 여부가 바뀌었습니다.**

원본인 `공용/매뉴얼/시스템매뉴얼.md`가 2회차에 개정되면서 기존에 완전 중복으로 묶여 탈락했던 복사본이 더 이상 완전 중복도, 근사 중복(임계값 `0.85`)도 아니게 되어 정상 문서로 살아남았기 때문입니다. 문서 단위 증분 처리는 '이번에 변경된 파일'만 다시 처리하므로 이처럼 **다른 문서의 개정으로 인해 중복 판정이 연쇄적으로 바뀐 문서**를 원리적으로 포착하지 못합니다.

결국 **중복 제거처럼 코퍼스 전체 집합을 대조해야 하는 단계는 문서 단위 증분만으로 완벽히 동기화할 수 없으며**, 그렇기 때문에 이론 5.1절과 5.4절에서 주기적인 전체 재처리와 증분-전체 대조 점검을 필수 절차로 둔 것입니다.

## Step 12. 반복 실행 기반 멱등성 검증

이론 2.6절에서 다룬 대로 멱등성은 코드 눈대중이 아니라 **동일한 상태에서 파이프라인을 두 번 연속 실행한 결과를 대조해 검증**합니다. 이때 **청크 식별자(`chunk_id`) 집합이 일치하는지**와 **동일 식별자의 본문(`text`) 내용이 일치하는지**를 모두 확인해야 합니다. 식별자만 같고 내부 본문이 매번 바뀌는 비결정적 오류는 개수만 봐서는 절대 드러나지 않기 때문입니다.

In [42]:
# 실습 : 멱등성을 검증하세요

def run_full():
    p = Pipeline(stage_dir="./stage_tmp")
    (p.add("catalog", lambda _: stage_catalog())
      .add("compliance", stage_compliance)
      .add("ingest", stage_ingest)
      .add("compliance_post", stage_compliance_post)
      .add("filter", stage_filter)
      .add("language", stage_language)
      .add("dedup", stage_dedup))
    d, _ = p.run([], verbose=False)
    return stage_chunk(d)


c1 = run_full()
c2 = run_full()

### START CODE HERE ### (청크 ID 집합과 ID->본문 딕셔너리 비교를 위한 4줄을 채우세요)
# 1) 1회차(c1)와 2회차(c2)의 chunk_id 집합(set)을 각각 만듭니다
ids1 = {c["chunk_id"] for c in c1}
ids2 = {c["chunk_id"] for c in c2}

# 2) 같은 chunk_id의 본문(text)까지 동일한지 확인하기 위해 {chunk_id: text} 딕셔너리를 만듭니다
t1 = {c["chunk_id"]: c["text"] for c in c1}
t2 = {c["chunk_id"]: c["text"] for c in c2}
### END CODE HERE ###

if ids1 is None or t1 is None:
    print("[안내] ids1, ids2, t1, t2 빈칸을 채운 뒤 실행하세요.")
else:
    print(f"1회차 청크 {len(c1)}개, 2회차 청크 {len(c2)}개")
    print(f"ID 집합 동일 : {ids1 == ids2}")
    if ids1 != ids2:
        print(f"  1회차만 : {ids1 - ids2}")
        print(f"  2회차만 : {ids2 - ids1}")
    print(f"본문 동일 : {t1 == t2}")
    print(f"collected_at 동일 : {c1[0]['collected_at'] == c2[0]['collected_at']}")

1회차 청크 24개, 2회차 청크 24개
ID 집합 동일 : True
본문 동일 : True
collected_at 동일 : False


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
ids1 = {c["chunk_id"] for c in c1}
ids2 = {c["chunk_id"] for c in c2}

t1 = {c["chunk_id"]: c["text"] for c in c1}
t2 = {c["chunk_id"]: c["text"] for c in c2}
```

ID 집합과 본문이 모두 같아야 멱등성이 성립합니다.

`collected_at` 비교는 일부러 넣었습니다. `collected_at`은 **매번 다르지만** 식별자 계산에 넣지 않았으므로 멱등성이 깨지지 않습니다. 이론 2.5절 첫 항목이 경고한 것은 시각을 기록하는 것이 아니라 **시각을 식별자에 섞는 것**입니다.

다르게 나오면 원인은 대개 시각이나 무작위 값이 ID에 섞였거나 딕셔너리나 집합의 순회 순서에 의존했거나 중복 제거의 대표 선택이 비결정적인 경우입니다.

대표 선택이 특히 미묘합니다. `pick_representative`의 점수가 동점이면 `max`가 어느 것을 고를지는 입력 순서에 달리는데 우리는 점수 마지막 기준으로 경로를 넣어 동점 자체를 없앴습니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 13. 중간 산출물(체크포인트) 기반 백필

이론 2.4절은 "다시 실행한다"를 방금 실패한 것을 곧바로 다시 하는 **재시도**, 멈춘 지점부터 이어 가는 **재개**, 실패해서가 아니라 의도해서 과거를 다시 처리하는 **백필**로 나눴습니다. 재시도는 Step 5의 `stage_ingest`에 넣었고 재개는 Step 13-1에서 실습합니다.

지금은 백필로 청크 길이를 바꿔 이미 처리한 문서를 새 설정으로 다시 만듭니다. 이론 2.4절에서 저장 지점을 **연산 비용이 큰 단계 바로 뒤**에 두라고 한 대로 저장 지점이 `dedup` 뒤에 있어 파싱과 중복 제거를 다시 하지 않아도 됩니다.

In [43]:
# dedup 단계의 산출물을 불러와 청킹만 다시 합니다
saved_docs = pipe2.load_stage("dedup")
print(f"저장된 문서 {len(saved_docs)}건 불러옴 (ok {sum(1 for d in saved_docs if d.status=='ok')}건)")

t0 = time.time()
for size in [60, 120, 400]:
    ch = stage_chunk(saved_docs, chunk_size=size)
    avg = np.mean([len(c["text"]) for c in ch]) if ch else 0
    longest = max((len(c["text"]) for c in ch), default=0)
    print(f"chunk_size {size:>4} : 청크 {len(ch):>3}개, 평균 {avg:>3.0f}자, 최장 {longest:>3}자")
print(f"\n세 번 합쳐 {time.time()-t0:.2f}초. 파싱도 중복 제거도 다시 하지 않았습니다")

저장된 문서 19건 불러옴 (ok 11건)
chunk_size   60 : 청크  38개, 평균  42자, 최장  60자
chunk_size  120 : 청크  28개, 평균  58자, 최장 116자
chunk_size  400 : 청크  24개, 평균  67자, 최장 228자

세 번 합쳐 0.00초. 파싱도 중복 제거도 다시 하지 않았습니다


최장 길이 열에서 보이듯 **실습 저장소의 조항이 짧아 120자 위로는 더 나눌 것이 없어서** 120과 400의 결과가 같게 나왔을 것입니다.

그래서 설정을 바꿀 때마다 결과를 측정합니다. 값을 400에서 800으로 올리고 "청크를 키웠다"고 보고했는데 실제로는 아무것도 달라지지 않는 일이 흔합니다. **바꿨다는 사실과 달라졌다는 사실은 다릅니다.**

설정을 바꿔 다시 만들었으면 이론 5.4절이 전체를 다시 처리해야 하는 때로 꼽은 **설정값을 바꿨다**에 해당합니다. 새 문서만 새 설정으로 처리하면 인덱스 안에 두 설정의 청크가 섞이므로 전체 재처리를 같이 계획하고 실행한 뒤에는 Step 11의 대조를 다시 합니다.

### Step 13-1. 중단 지점 이후 단계 재개(Resume)

이론 1.2절에서 본 대로 문서 1만 건을 파싱하는 데 세 시간이 걸리는데 8,200건째에서 멈추면 그 세 시간이 통째로 사라집니다. 여기서는 이론 2.4절의 **재개**를 단계 단위로 구현합니다(8,200건째부터 이어 가려면 문서마다 저장 지점이 필요합니다). 비싼 파싱 뒤에 저장 지점을 두고 그 뒤 단계에서 멈췄을 때 파싱을 다시 하지 않고 이어 가는지, 그 결과가 처음부터 실행한 결과와 같은지 확인합니다. 이 실습은 자기 저장 폴더(`./stage_resume`, `./stage_resume2`)만 쓰므로 앞 Step의 산출물을 바꾸지 않습니다.

In [44]:
# 실습 : 파이프라인 중간에 실패를 발생시키고, 저장된 단계부터 재개(Resume)하세요

FAIL_ONCE = {"done": False}


def stage_flaky_language(docs):
    if not FAIL_ONCE["done"]:
        FAIL_ONCE["done"] = True
        raise RuntimeError("언어 감지 서비스 일시 장애")
    return stage_language(docs)


pipe_r = Pipeline(stage_dir="./stage_resume")
(pipe_r
 .add("catalog", lambda _: stage_catalog(), save=True)
 .add("compliance", stage_compliance, save=True)
 .add("ingest", stage_ingest, save=True)          # 연산 비용이 큰 단계 바로 뒤에 저장
 .add("language", stage_flaky_language)
 .add("dedup", stage_dedup))

partial, ok = pipe_r.run([])
print(f"완료 여부 {ok}, 멈춘 단계 {pipe_r.stopped_at}")
print(f"저장된 단계 : {sorted(p.stem for p in Path('./stage_resume').glob('*.json'))}")

### START CODE HERE ### (저장된 산출물 복원 및 재개 2줄을 채우세요)
# 1) 멈춘 단계(language) 바로 앞 단계인 "ingest"의 저장된 산출물을 불러옵니다
saved = pipe_r.load_stage("ingest")

# 2) 복원한 산출물(saved)을 입력으로 넣고 멈춘 단계(start_from=pipe_r.stopped_at)부터 재개합니다
resumed, ok2 = pipe_r.run(saved, start_from=pipe_r.stopped_at)
### END CODE HERE ###

if saved is None or resumed is None:
    print("\n[안내] saved 와 resumed, ok2 빈칸을 채운 뒤 다시 실행하세요.")
else:
    print(f"\ningest 산출물 {len(saved)}건 복원. 파싱을 다시 하지 않습니다")
    print(f"\n재개 완료 {ok2}. ok 상태 {sum(1 for d in resumed if d.status=='ok')}건")

    FAIL_ONCE["done"] = True
    fresh = Pipeline(stage_dir="./stage_resume2")
    (fresh.add("catalog", lambda _: stage_catalog())
          .add("compliance", stage_compliance)
          .add("ingest", stage_ingest)
          .add("language", stage_language)
          .add("dedup", stage_dedup))
    whole, _ = fresh.run([], verbose=False)
    print(f"재개 결과 == 통째 실행 결과 : "
          f"{sorted(d.doc_id for d in resumed if d.status=='ok') == sorted(d.doc_id for d in whole if d.status=='ok')}")

[catalog         ] 입력   0 -> 출력  19 (제외 0, 실패 0) 0.04초
[compliance      ] 입력  19 -> 출력  18 (제외 1, 실패 0) 0.02초
[ingest          ] 입력  18 -> 출력  15 (제외 4, 실패 0) 0.05초
[language] 단계가 중단되었습니다 : RuntimeError 언어 감지 서비스 일시 장애
완료 여부 False, 멈춘 단계 language
저장된 단계 : ['catalog', 'compliance', 'ingest']
[건너뜀] catalog
[건너뜀] compliance
[건너뜀] ingest
[language        ] 입력  15 -> 출력  14 (제외 5, 실패 0) 0.02초
[dedup           ] 입력  14 -> 출력  12 (제외 7, 실패 0) 0.05초

ingest 산출물 19건 복원. 파싱을 다시 하지 않습니다

재개 완료 True. ok 상태 12건
재개 결과 == 통째 실행 결과 : True


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
saved = pipe_r.load_stage("ingest")
resumed, ok2 = pipe_r.run(saved, start_from=pipe_r.stopped_at)
```

`run(saved, start_from=...)`이 앞 세 단계를 건너뛰는 것을 출력에서 확인합니다. 건너뛴 단계는 `[건너뜀]`으로 출력됩니다.

재개 결과와 통째 실행 결과의 비교를 봅니다. **재개한 결과가 처음부터 실행한 결과와 같아야** 재개가 쓸모 있습니다. 같지 않으면 재개는 시간을 아끼는 대신 다른 답을 내는 장치가 되고 이론 5.4절에서 말한 어긋남이 쌓이는 지점이 됩니다.

저장 지점을 어디에 둘지는 설계 결정입니다. 모든 단계를 저장하면 디스크와 시간이 낭비되고 너무 적게 저장하면 재개 효과가 없습니다. **연산 비용이 큰 단계 바로 뒤**가 원칙이고 이론 2.4절은 전체 실행이 10분을 넘기 시작하면 저장 지점을 만들라고 했습니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 14. 정기 스케줄 등록 및 무실행 감시

여기까지가 이론 1.6절의 여덟 줄 중 일곱 줄이고 남은 하나는 정해진 시각에 실행되도록 스케줄에 등록하는 일입니다. 이 일은 반드시 마지막에 합니다. **계측도 게이트도 없는 처리를 먼저 정기 실행에 등록하면 조용히 망가진 결과가 실행할 때마다 인덱스에 들어갑니다.** 이 노트의 파이프라인은 매주 월요일 새벽 3시에 실행됩니다.

이론 3.6절에서 첫 파이프라인에는 오케스트레이션 도구가 필요 없다고 했습니다. 오케스트레이션 도구가 해 주는 넷 중 첫째인 스케줄 실행만 있으면 되고 그것은 운영체제에 이미 들어 있습니다.

**[현업 환경 적용]** : 코랩은 탭을 닫으면 런타임이 끊겨 스케줄 실행을 등록할 환경이 아니므로 아래 한 줄은 코랩에서 실행할 명령이 아닙니다. 사내 리눅스 서버에서 `crontab -e`로 등록하는 한 줄을 만들어 보는 셀입니다.

In [45]:
# 🖥 로컬 작업 : 사내 서버에서 등록할 한 줄을 조립해 봅니다
CRON_LINE = ("0 3 * * 1  cd /srv/rag && /srv/rag/.venv/bin/python run_pipeline.py"
             "  >> /var/log/rag/pipeline.log 2>&1")

print(CRON_LINE)
print()
print("앞의 다섯 필드 : 분 시 일 월 요일")
print("0 3 * * 1  ->  매주 월요일 새벽 3시")
print()
print("눈여겨볼 곳")
print("  - 절대 경로를 씁니다. 스케줄 실행은 여러분의 셸 설정을 물려받지 않습니다")
print("  - 가상환경의 파이썬을 직접 가리킵니다")
print("  - 출력을 로그 파일로 넘깁니다. 실행 기록이 남는 위치입니다")

0 3 * * 1  cd /srv/rag && /srv/rag/.venv/bin/python run_pipeline.py  >> /var/log/rag/pipeline.log 2>&1

앞의 다섯 필드 : 분 시 일 월 요일
0 3 * * 1  ->  매주 월요일 새벽 3시

눈여겨볼 곳
  - 절대 경로를 씁니다. 스케줄 실행은 여러분의 셸 설정을 물려받지 않습니다
  - 가상환경의 파이썬을 직접 가리킵니다
  - 출력을 로그 파일로 넘깁니다. 실행 기록이 남는 위치입니다


이론 3.6절의 다섯 신호 가운데 **하나도 걸리지 않으면 crontab 한 줄로 충분합니다.** 두세 개가 걸리면 그때 Airflow 같은 오케스트레이션 도구를 검토합니다. 오늘 만든 `stage_*` 함수들은 이론 3.1절의 원칙대로 도구와 무관한 평범한 파이썬이라 나중에 어느 도구로 옮기든 이 함수들을 호출하기만 하면 됩니다.

이론 6.5절에서 감시는 "실패하면 알려라"가 아니라 "**정해진 시각까지 성공 기록이 없으면 알려라**"로 걸라고 했습니다. 스케줄 등록이 지워졌거나 서버가 꺼졌으면 실패조차 생기지 않으므로 `runs/` 폴더의 마지막 기록 시각을 보는 감시가 그 역할을 합니다.

## Step 15. 선택 심화 트랙

트랙 1, 2, 3, 5 가운데 **두 개를 선택**해 수행하고 공유합니다. (트랙 4 실패 재개 시나리오는 본 실습 Step 13-1로 이동했습니다.)

<div style="font-size: 18px; line-height: 1.6;">

| 트랙 | 주제 | 성격 | 난이도 |
|---|---|---|---|
| 1 | 경보 3단계 분류와 소프트 게이트 승격 | 확장 | 중 |
| 2 | 단계별 병목 측정과 병렬화 | 최적화 | 중 |
| 3 | 벡터DB 연동과 증분 반영 | 확장 | 상 |
| 4 | 권한 기반 인덱스 분리 | 설계 | 상 |

</div>

트랙 5(권한 기반 인덱스 분리)는 트랙 3의 컬렉션과 임베딩 모델을 그대로 사용하므로 트랙 5를 선택할 경우 트랙 3을 먼저 수행합니다.

#### 트랙 1. 경보 3단계 분류와 연속 발생 시 등급 승격

Step 8에서 붙인 소프트 게이트가 걸릴 때마다 대화방으로 보내면 이론 6절 두 번째 Quiz가 경고한 대로 며칠 만에 아무도 안 읽습니다.

경보를 이론 6.5절의 세 단계로 나누고 같은 경고가 연속으로 걸리면 하드로 올리는 운영 규칙을 만듭니다.

In [46]:
# 실습 : 경고를 세 단계(즉시 / 아침요약 / 기록만)로 분류하고 연속 발생 횟수를 집계하세요

ALERT_RULES = [
    # (판정 함수, 단계, 조치 안내)
    (lambda st: st["n_out"] == 0, "즉시", "인덱스가 비워집니다. 원본 경로와 권한을 먼저 보세요"),
    (lambda st: st["n_failed"] > 0, "아침요약", "실패 사유별 건수를 확인하세요"),
    (lambda st: bool(st["warnings"]), "기록만", "직전 실행 대비 변화입니다. 3회 연속이면 승격을 검토합니다"),
]


def classify(stage_stat):
    for test, level, action in ALERT_RULES:
        if test(stage_stat):
            return level, action
    return None, ""


def consecutive_warnings(stage_name, limit=5):
    """최근 실행 기록에서 같은 단계의 경고가 몇 회 연속인지 셉니다"""
    files = sorted(RUN_DIR.glob("run_*.json"))[-limit:]
    streak = 0
    for f in reversed(files):
        rec = json.load(open(f, encoding="utf-8"))
        st = rec["stages"].get(stage_name)
        ### START CODE HERE ### (경고가 있으면 streak를 1 늘리고, 없으면 반복을 멈추세요)
        if st and st["warnings"]:
            streak += 1
        else:
            break
        ### END CODE HERE ###
    return streak


last = load_last_run()
print(f"{'단계':<18} {'분류':<8} {'연속':>4}  조치 안내")
print("-" * 82)
for name, st in last["stages"].items():
    level, action = classify(st)
    streak = consecutive_warnings(name)
    print(f"{name:<18} {level or '없음':<8} {streak:>4}  {action or '조치 불필요'}")

print()
promote = [n for n in last["stages"] if consecutive_warnings(n) >= 3]
if promote:
    for n in promote:
        print(f"[승격 제안] {n} 소프트 게이트가 3회 이상 연속 걸렸습니다. "
              f"하드 게이트로 올리는 것을 검토하세요")
else:
    print("승격 제안 없음. 3회 연속 경고가 쌓인 단계가 없습니다")

단계                 분류         연속  조치 안내
----------------------------------------------------------------------------------
catalog            없음          0  조치 불필요
compliance         없음          0  조치 불필요
ingest             없음          0  조치 불필요
compliance_post    없음          0  조치 불필요
filter             없음          0  조치 불필요
language           없음          0  조치 불필요
dedup              없음          0  조치 불필요

승격 제안 없음. 3회 연속 경고가 쌓인 단계가 없습니다


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
if st and st["warnings"]:
    streak += 1
else:
    break
```

`ALERT_RULES`는 리스트로 둡니다. 위에서부터 보고 처음 걸리는 규칙의 단계를 쓰므로 **가장 센 것을 맨 위에 둡니다.**

모든 규칙에는 이론 6.5절의 조언대로 **조치 안내**가 붙어 있습니다. 이 항목이 있어야 처음 받는 사람도 대응할 수 있습니다.

`consecutive_warnings`는 실행 기록이 쌓여 있어야 의미가 있습니다. Step 9와 Step 10에서 만들어 둔 `runs/`가 여기서 쓰입니다. **감시는 기록 위에서만 가능합니다.**

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 2. 단계별 병목 측정과 파싱 병렬화

파이프라인 리포트의 소요 시간 열로 어느 단계가 병목인지 찾고 개선합니다.

In [47]:
# 실습 : 문서를 200건 규모로 복제해 단계별 시간을 측정하고, 파싱 단계를 병렬화하세요

from concurrent.futures import ThreadPoolExecutor

BIG = Path("./corp_docs_big")
if BIG.exists():
    shutil.rmtree(BIG)
BIG.mkdir()
src_files = sorted(p for p in ROOT.rglob("*") if p.is_file())
for i in range(10):  # 실습용 80건 규모로 경량화
    for p in src_files:
        rel = p.relative_to(ROOT)
        dst = BIG / f"copy{i}" / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            t = p.read_text(encoding="utf-8")
            dst.write_text(f"# 사본 {i}\n\n{t}", encoding="utf-8")
        except UnicodeDecodeError:
            shutil.copy(p, dst)

print(f"확대된 저장소 : {sum(1 for _ in BIG.rglob('*') if _.is_file())}건")

ORIG_ROOT = ROOT
ROOT = BIG

pipe_big = Pipeline(stage_dir="./stage_big")
(pipe_big
 .add("catalog", lambda _: stage_catalog())
 .add("compliance", stage_compliance)
 .add("ingest", stage_ingest)
 .add("compliance_post", stage_compliance_post)
 .add("filter", stage_filter)
 .add("language", stage_language)
 .add("dedup", stage_dedup))
db, _ = pipe_big.run([], verbose=False)
pipe_big.report()


def stage_ingest_parallel(docs, workers=8):
    targets = [d for d in docs if d.status in ("pending", "ok") and d.ext in PARSERS]

    def work(d):
        try:
            text, enc = PARSERS[d.ext](ROOT / d.path)
            d.text = text
            d.log("ingest", f"파싱 완료 ({enc})")
        except Exception as e:
            d.fail("ingest", f"형식오류: {type(e).__name__}")
        return d

    ### START CODE HERE ### (ThreadPoolExecutor로 targets에 work 함수를 병렬 매핑하세요)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(work, targets))
    ### END CODE HERE ###

    for d in docs:
        if d.status in ("pending", "ok") and d.ext not in PARSERS:
            d.exclude("ingest", f"미지원형식: {d.ext}")
    return docs


for fn, label in [(stage_ingest, "순차"), (stage_ingest_parallel, "병렬")]:
    ds = stage_catalog()
    ds = stage_compliance(ds)
    t0 = time.time()
    fn(ds)
    print(f"{label} 파싱 : {time.time()-t0:.2f}초")

ROOT = ORIG_ROOT

확대된 저장소 : 190건
단계                     입력     출력     제외     실패       초
--------------------------------------------------------
catalog                 0    190      0      0    0.36
compliance            190      0    190      0    0.00
ingest                  0      0    190      0    0.00
compliance_post         0      0    190      0    0.00
filter                  0      0    190      0    0.00
language                0      0    190      0    0.00
dedup                   0      0    190      0    0.02
순차 파싱 : 0.00초
병렬 파싱 : 0.00초


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
with ThreadPoolExecutor(max_workers=workers) as ex:
    list(ex.map(work, targets))
```

파일 입출력과 CPU 연산이 섞여 있어 파싱과 언어 감지가 병목으로 나오는 경우가 많습니다.

**그런데 이 실습에서는 병렬 쪽이 더 느리게 나올 수 있습니다.** 파일이 작아 한 건 읽는 시간보다 작업을 나눠 주는 비용이 커서 생기는 측정 결과이지 실패가 아닙니다.

**효과가 없다는 것도 측정해서 알아야 하고** 측정하지 않고 병렬화부터 하면 복잡한 코드만 남습니다. 파일 크기를 키우거나 원격 저장소에서 읽는 상황으로 바꿔 다시 측정해 봅니다.

주의할 점도 있습니다. 문서 집합 전체를 봐야 하는 **중복 제거는 병렬화가 어렵습니다.** 단계의 성격에 따라 병렬화 가능성이 다릅니다.

또 병렬화하면 **순서가 비결정적이 되어 멱등성이 깨질 수 있습니다.** 위 구현에서 `ex.map`은 순서를 보존하지만 대표 선택 같은 집합 연산은 정렬된 입력을 전제로 해야 안전합니다. 이론 2.5절 둘째 항목입니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 3. 벡터 데이터베이스(Qdrant) 연동과 증분 반영

파이프라인의 출력을 실제 인덱스에 반영합니다. `stale_chunk_ids`를 지우고 새 청크를 넣습니다.

In [48]:
# 실습 : Qdrant에 청크를 결정적 ID로 덮어쓰기(Upsert)하고, 증분 조정(stale_chunk_ids 삭제)을 반영하세요

import torch
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from langchain_huggingface import HuggingFaceEmbeddings

EMB_MODEL = "intfloat/multilingual-e5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

emb = HuggingFaceEmbeddings(
    model_name=EMB_MODEL,
    model_kwargs={"device": DEVICE, "model_kwargs": {"dtype": DTYPE}},
    encode_kwargs={"normalize_embeddings": True})

DIM = len(emb.embed_query("차원 확인"))
print(f"{EMB_MODEL}  device={DEVICE}  dim={DIM}")

COLL = "corp_docs"
client = QdrantClient(path="./qdrant_pipeline")

if not client.collection_exists(COLL):
    client.create_collection(
        collection_name=COLL,
        vectors_config=VectorParams(size=DIM, distance=Distance.COSINE))


def to_point_id(chunk_id):
    return int(chunk_id[:12], 16)


def upsert_chunks(chunks, batch=64):
    if not chunks:
        return 0
    n = 0
    for i in range(0, len(chunks), batch):
        part = chunks[i:i+batch]
        vecs = emb.embed_documents([f"passage: {c['text']}" for c in part])
        ### START CODE HERE ### (결정적 포인트 ID인 to_point_id(c["chunk_id"])를 지정해 PointStruct 생성)
        points = [PointStruct(id=to_point_id(c["chunk_id"]), vector=v, payload=c)
                  for c, v in zip(part, vecs)]
        ### END CODE HERE ###
        if points and points[0].id is None:
            print("[안내] PointStruct의 id 빈칸을 채운 뒤 다시 실행하세요.")
            return 0
        client.upsert(collection_name=COLL, points=points)
        n += len(points)
    return n


def delete_chunks(chunk_ids):
    """삭제 요청 건수가 아니라 실제로 인덱스에서 줄어든 포인트 수를 반환합니다"""
    if not chunk_ids:
        return 0
    before = client.count(COLL).count
    client.delete(collection_name=COLL,
                  points_selector=[to_point_id(cid) for cid in chunk_ids])
    return before - client.count(COLL).count


# 1회차 결과를 먼저 넣습니다
upsert_chunks(chunks)
print(f"1회차 인덱스 포인트 수 {client.count(COLL).count}")

# 같은 청크를 다시 넣어 멱등성 확인
upsert_chunks(chunks)
print(f"재인덱싱 후 포인트 수 {client.count(COLL).count}")

# 2회차 청크를 넣습니다 (아직 구버전이 남아 있어 수가 늘어납니다)
upsert_chunks(chunks2)
print(f"2회차를 넣은 뒤 포인트 수 {client.count(COLL).count}")

### START CODE HERE ### (diff["stale_chunk_ids"]를 넘겨 구버전 청크를 삭제하세요)
removed = delete_chunks(diff["stale_chunk_ids"])
### END CODE HERE ###

final = client.count(COLL).count
print(f"조정으로 지운 청크 {removed}개, 최종 포인트 수 {final}")
print(f"전체 재처리 청크 수 {len(chunks2)}개와 일치 : {final == len(chunks2)}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

intfloat/multilingual-e5-small  device=cpu  dim=384
1회차 인덱스 포인트 수 19
재인덱싱 후 포인트 수 19
2회차를 넣은 뒤 포인트 수 27
조정으로 지운 청크 3개, 최종 포인트 수 24
전체 재처리 청크 수 24개와 일치 : True


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 결정적 포인트 ID로 PointStruct 생성
points = [PointStruct(id=to_point_id(c["chunk_id"]), vector=v, payload=c)
          for c, v in zip(part, vecs)]

# 2) 스냅샷 비교로 찾은 구버전 청크(stale_chunk_ids) 삭제
removed = delete_chunks(diff["stale_chunk_ids"])
```

먼저 **두 번 넣어도 포인트 수가 같은지** 확인합니다. `chunk_id`가 결정적이므로 같은 ID에 덮어쓰기가 되고 이론 2.3절의 두 번째 장치인 **넣기가 아니라 덮어쓰기로 쓰기**를 그대로 보여 줍니다.

다음으로 **2회차 청크를 넣은 직후 포인트 수가 늘어나는 것**을 봅니다. 이 순간의 인덱스에는 구버전과 신버전이 함께 있고 여기서 멈추면 이론 2.2절의 두 번째 사고가 됩니다. 조정이 그 뒤를 마무리합니다.

`delete_chunks`는 요청 건수가 아니라 **실제로 줄어든 수**를 반환합니다. 요청 건수를 그대로 출력하면 지울 것이 인덱스에 없었을 때도 "삭제 3개"라고 보고하게 됩니다. **지우는 일은 빠뜨려도 눈에 안 보이므로 건수로 확인해야 합니다.**

`client.recreate_collection`은 쓰지 않았습니다. 최신 `qdrant-client`에서 폐기 예정입니다. 또 이름 그대로 **있던 것을 지우고 새로 만들어** 매 실행마다 인덱스를 통째로 비우므로 증분 반영과 정반대로 동작합니다.

차원도 384로 고정하지 않고 `emb.embed_query`의 길이에서 읽어 모델을 바꾸면 차원이 따라 바뀝니다.

`int(cid[:12], 16)` 변환은 해시의 앞부분만 쓰므로 이론적으로 충돌할 수 있습니다. 실무에서는 UUID를 쓰거나 해시 전체를 담을 수 있는 방식을 택합니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 4. 실패 재개 시나리오
*(안내: 이 트랙은 본 실습 **Step 13-1**로 이동하여 필수 과정으로 진행했습니다.)*

#### 트랙 5. 접근 권한 기반 검색 필터링과 컬렉션 분리

Step 4에서 제외한 기밀 문서를 **분리**하는 설계로 바꿉니다. 트랙 3의 `client`와 `emb`를 그대로 씁니다.

In [49]:
# 실습 : 접근 등급(access_level)별로 검색 범위를 제한하는 권한 필터를 완성하세요

from qdrant_client.models import Filter, FieldCondition, MatchAny

ROLE_CLEARANCE = {
    "employee": ["public", "draft"],
    "manager": ["public", "draft", "internal"],
    "hr_admin": ["public", "draft", "internal", "confidential"],
}


def stage_compliance_split(docs):
    """제외하지 않고 등급과 방침만 부여합니다"""
    for d in docs:
        if d.status not in ("pending", "ok"):
            continue
        if d.ext in (".md", ".txt"):
            pii = detect_pii(read_bytes_text(ROOT / d.path)[0])
            d.policy = "mask" if pii else "include"
        else:
            d.policy = ""
        d.log("compliance", f"등급 {d.access_level}, 방침 {d.policy}")
    return docs


pipe_s = Pipeline(stage_dir="./stage_split")
(pipe_s
 .add("catalog", lambda _: stage_catalog())
 .add("compliance", stage_compliance_split)
 .add("ingest", stage_ingest)
 .add("compliance_post", stage_compliance_post)
 .add("filter", stage_filter)
 .add("language", stage_language)
 .add("dedup", stage_dedup))
docs_s, _ = pipe_s.run([], verbose=False)
chunks_s = stage_chunk(docs_s)
print(f"등급별 청크 : {Counter(c['access_level'] for c in chunks_s)}")

COLL_S = "corp_docs_acl"
if not client.collection_exists(COLL_S):
    client.create_collection(
        collection_name=COLL_S,
        vectors_config=VectorParams(size=DIM, distance=Distance.COSINE))

vecs = emb.embed_documents([f"passage: {c['text']}" for c in chunks_s])
client.upsert(collection_name=COLL_S,
              points=[PointStruct(id=to_point_id(c["chunk_id"]), vector=v, payload=c)
                      for c, v in zip(chunks_s, vecs)])
print(f"인덱싱 {client.count(COLL_S).count}개")


def search_with_clearance(query, role, top_k=3):
    ### START CODE HERE ### (역할별 허용 등급 목록 조회 및 Qdrant MatchAny 필터 작성)
    # 1) ROLE_CLEARANCE 에서 role 에 해당하는 허용 등급 리스트를 가져옵니다
    allowed = ROLE_CLEARANCE[role]

    # 2) key="access_level" 필드가 allowed 중 하나와 일치(MatchAny(any=allowed))하는 Filter를 만듭니다
    acl_filter = Filter(must=[FieldCondition(key="access_level", match=MatchAny(any=allowed))])
    ### END CODE HERE ###

    if allowed is None or acl_filter is None:
        return []

    hits = client.query_points(
        collection_name=COLL_S,
        query=emb.embed_query(f"query: {query}"),
        query_filter=acl_filter,
        limit=top_k,
    ).points
    return [(h.payload["source_path"], h.payload["access_level"], h.score) for h in hits]


TOTAL = client.count(COLL_S).count
conf_paths = {c["source_path"] for c in chunks_s if c["access_level"] == "confidential"}
print(f"기밀 등급 문서 : {sorted(conf_paths)}\n")

for role in ["employee", "manager", "hr_admin"]:
    rows = search_with_clearance("인사평가 결과", role, top_k=TOTAL)
    if not rows:
        print(f"[{role}] 빈칸(allowed, acl_filter)을 채운 뒤 실행하세요.")
        break
    got_levels = Counter(level for _, level, _ in rows)
    leak = [r for r in rows if r[1] not in ROLE_CLEARANCE[role]]
    conf_hit = [r for r in rows if r[0] in conf_paths]
    print(f"[{role}] 허용 등급 {ROLE_CLEARANCE[role]}")
    print(f"  꺼낸 청크 {len(rows)}/{TOTAL}  등급 분포 {dict(got_levels)}")
    print(f"  권한 밖 유출 {len(leak)}건, 기밀 문서 접근 {len(conf_hit)}건")
    print(f"  상위 2건 : {[(p, l) for p, l, _ in rows[:2]]}")

등급별 청크 : Counter({'public': 20, 'draft': 3, 'internal': 1, 'confidential': 1})
인덱싱 25개
기밀 등급 문서 : ['인사/평가/2026상반기_인사평가.md']

[employee] 허용 등급 ['public', 'draft']
  꺼낸 청크 23/25  등급 분포 {'public': 20, 'draft': 3}
  권한 밖 유출 0건, 기밀 문서 접근 0건
  상위 2건 : [('공용/부서별예산.xlsx', 'public'), ('공용/매뉴얼/비상연락망.md', 'public')]
[manager] 허용 등급 ['public', 'draft', 'internal']
  꺼낸 청크 24/25  등급 분포 {'public': 20, 'internal': 1, 'draft': 3}
  권한 밖 유출 0건, 기밀 문서 접근 0건
  상위 2건 : [('공용/부서별예산.xlsx', 'public'), ('공용/매뉴얼/비상연락망.md', 'public')]
[hr_admin] 허용 등급 ['public', 'draft', 'internal', 'confidential']
  꺼낸 청크 25/25  등급 분포 {'confidential': 1, 'public': 20, 'internal': 1, 'draft': 3}
  권한 밖 유출 0건, 기밀 문서 접근 1건
  상위 2건 : [('인사/평가/2026상반기_인사평가.md', 'confidential'), ('공용/부서별예산.xlsx', 'public')]


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
allowed = ROLE_CLEARANCE[role]
acl_filter = Filter(must=[FieldCondition(
    key="access_level", match=MatchAny(any=allowed))])
```

**employee로는 기밀 문서 접근이 0건이고 hr_admin으로는 0건이 아니어야** 합니다. 둘 다 확인해야 필터가 실제로 동작하는 것이고 한쪽만 보면 그냥 아무것도 안 나오는 상태와 구분되지 않습니다.

같은 이유로 `top_k`를 인덱스 전체 크기로 잡았습니다. 상위 3건만 보면 기밀 문서가 안 나온 이유가 **권한 때문인지 점수가 낮아서인지** 구분되지 않으므로 권한 검증은 순위에 기대지 않습니다.

이 설계에도 위험이 있습니다. 필터를 코드에서 붙이므로 **필터를 빼먹으면 즉시 유출됩니다.** 그래서 실무에서는 설계를 보강합니다.

첫째, 등급별로 **컬렉션을 물리적으로 분리**합니다. 코드 실수로 필터가 빠져도 다른 컬렉션의 데이터는 나오지 않습니다.

둘째, `ROLE_CLEARANCE[role]`처럼 **권한 인자에 기본값을 두지 않습니다.** 기본값을 public으로 두면 인자를 빠뜨렸을 때 조용히 실행되고 필수로 두면 바로 실패합니다. 실패가 유출보다 낫습니다.

이 주제는 보안 주간에서 접근 제어와 함께 다룹니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 16. 실습 결과 점검과 정리

각자 수행한 두 트랙을 공유합니다. 아래 형식으로 정리해주세요.
1. **선택한 트랙과 이유**
2. **측정 결과** : 검색 정확도(`Golden Set` hit@1·hit@3, 멱등성·증분 대조 일치율)와 비용(단계별 실행 시간, 병렬화 단축률, 재개 시 절감 시간, 증분 업서트·삭제 건수)의 변화
3. **단일 스크립트 기반 파이프라인과 스냅샷 증분 동기화의 한계를 만난 지점** : 구버전 규정 공존으로 인한 hit@1 하락, 무음 장애(0건 수집·폴더 누락), 권한 필터링 시 순위 왜곡 등 구체적으로
4. **이 시스템을 사내에 배포한다면** : 주기적 배치에서 문서 몇 건까지 감당 가능한지, 무엇이 병목인지

공유가 끝나면 오늘 만든 것을 한 문장으로 정리해봅시다.
우리는 내용 해시로 몇 번을 돌려도 같은 결과가 나오는 식별자를 만들고, 중간에 멈춰도 이어 달리는 재개 장치를 두고, 모든 청크에 출처 계보를 새기고, 0건 수집이나 구버전 혼입 같은 무음 장애를 막는 검증 게이트를 세웠습니다.
여기서 질문이 하나 생깁니다. 이 모든 것을 통틀어 부르는 이름이 있을까요? 위에서 아래로 한 번 실행하고 끝나는 주피터 노트북도 아니고 단순한 예약 작업(`cron`)도 아니고, 데이터의 흐름과 상태·품질을 스스로 통제하는 이 실행 구조 전체를 말입니다. 내일 벡터 데이터베이스 노트(`D43`)에서는 이 파이프라인이 쏟아내는 청크를 담아 무중단으로 서빙하는 **온프레미스 벡터DB 엔지니어링**으로 넘어갑니다.

<div style="font-size: 18px; line-height: 1.6;">

| 이론 1.5절 용어 | 우리가 만든 것 |
|---|---|
| 멱등성 | 내용 해시로 만든 결정적 `doc_id`와 `chunk_id` |
| 재개 | 중간 산출물 저장과 `Pipeline.run(start_from=...)` |
| 리니지 | `Doc.history`, 청크의 여덟 항목, `runs/`의 실행 기록 |
| 증분 처리 | 스냅샷 대조와 조정, 전체 재처리와의 대조 |
| 검증 게이트 | 하드 게이트 셋과 소프트 게이트 하나 |

</div>

<div style="font-size: 18px; line-height: 1.6;">

| 이론 1.5절 용어 | 우리가 만든 것 |
|---|---|
| 멱등성 | 내용 해시로 만든 결정적 `doc_id`와 `chunk_id` |
| 재개 | 중간 산출물 저장과 `Pipeline.run(start_from=...)` |
| 리니지 | `Doc.history`, 청크의 여덟 항목, `runs/`의 실행 기록 |
| 증분 처리 | 스냅샷 대조와 조정, 전체 재처리와의 대조 |
| 검증 게이트 | 하드 게이트 셋과 소프트 게이트 하나 |

</div>

## 마무리 과제. 10단계 운영 파이프라인 구축

여러분의 문서 모음으로 열 단계 파이프라인을 구축합니다.

**1단계. 파이프라인 구현**

1. 문서 20건 이상, 형식 3종 이상으로 저장소를 만듭니다.  
2. 열 단계 중 최소 8단계를 구현합니다. 생략한 단계와 그 이유를 적습니다.  
3. 각 단계에 계측을 붙입니다. 입력, 출력, 제외 건수와 사유별 분포.  
4. 하드 게이트 2개, 소프트 게이트 1개 이상을 설계하고 각각을 하드/소프트로 정한 근거를 적습니다.

**2단계. 요구사항 검증**

5. **멱등성** : 두 번 실행해 청크 ID와 본문이 동일한지 확인합니다.  
6. **재개** (Step 13-1) : 중간 단계에서 실패를 발생시키고 재개해 앞 단계를 다시 하지 않음을 확인합니다. 그리고 재개 결과가 통째 실행 결과와 같은지도 확인합니다.  
7. **리니지** : 임의의 청크 하나를 골라 원본까지 역추적한 기록을 첨부합니다.

**3단계. 증분 처리**

8. 문서를 추가, 수정, 삭제하고 파이프라인을 다시 실행합니다.  
9. 조정 결과와 인덱스에서 제거할 청크 목록이 올바르게 나오는지 확인합니다.  
10. 전체 재처리 결과와 증분 결과를 대조해 양쪽에만 있는 청크가 몇 개인지 적습니다.

**4단계. 리포트**

- **단계별 리포트 표** : 입출력, 제외 사유, 소요 시간  
- **정답을 아는 질의 결과** (Step 9-1) : 자기 문서로 만든 질의 5개 이상과 기대 문서, hit@3과 hit@1, 실패한 질의와 원인  
- **병목 단계와 개선 시도** : 가장 느린 단계(Step 8 리포트의 초 열)와 개선 방법, 결과. 개선 시도는 선택(트랙 2)  
- **게이트 설계 근거** : 각 게이트를 하드/소프트로 정한 이유. 특히 "틀린 데이터와 갱신 지연 중 무엇이 더 나쁜가"에 대한 판단  
- **멱등성이 깨진 경험** : 처음부터 성립했다면 어떤 설계 덕분인지, 깨졌다면 원인과 수정  
- **리니지로 발견한 문제** : 역추적을 통해 알게 된 파이프라인의 결함

**평가 기준**

파이프라인이 실행되는지가 아니라 **운영 가능성**으로 평가합니다. 멱등성이 성립하는지, 실패를 재개할 수 있는지, 제외된 문서를 추적할 수 있는지, 그리고 게이트 설계에 판단 근거가 있는지를 봅니다. 계측 없는 파이프라인은 스크립트입니다.